In [3]:
import sys, logging
sys.path.insert(0, '.')
logging.basicConfig(level=logging.INFO)
import pandas as pd

w = pd.read_csv('Smarthack/gridpilot-ai/data/processed/dataset.csv')
print(w.shape)
print(w.columns.tolist())
print(w.groupby('segment_id').size())
print(w.timestamp_utc.min(), w.timestamp_utc.max())
print(w.describe().T)

(105165, 35)
['timestamp_utc', 'span_id', 'target_timestamp_utc', 'air_temp_c', 'wind_speed_ms', 'wind_from_deg', 'solar_ghi_wm2', 'wind_dir_sin', 'wind_dir_cos', 'air_temp_c_lag_1h', 'wind_speed_ms_lag_1h', 'wind_dir_sin_lag_1h', 'wind_dir_cos_lag_1h', 'solar_ghi_wm2_lag_1h', 'air_temp_c_lag_3h', 'wind_speed_ms_lag_3h', 'wind_dir_sin_lag_3h', 'wind_dir_cos_lag_3h', 'solar_ghi_wm2_lag_3h', 'air_temp_c_lag_6h', 'wind_speed_ms_lag_6h', 'wind_dir_sin_lag_6h', 'wind_dir_cos_lag_6h', 'solar_ghi_wm2_lag_6h', 'hour_utc', 'month_utc', 'target_air_temp_c_1h', 'target_wind_speed_ms_1h', 'target_wind_from_deg_1h', 'target_solar_ghi_wm2_1h', 'latitude', 'longitude', 'era5_latitude', 'era5_longitude', 'split']


KeyError: 'segment_id'

In [8]:
weather = pd.DataFrame({
    'timestamp_utc': w['timestamp_utc'],
    'segment_id': w['span_id'].astype(str),
    'air_temp_c': w['air_temp_c'],
    'wind_speed_ms': w['wind_speed_ms'],
    'wind_direction_deg': w['wind_from_deg'],
    'solar_radiation': w['solar_ghi_wm2'],
    'source': 'dataset.csv',
    'data_quality': 'unknown',   # datasetul nu are o coloană de calitate
})
weather.to_csv('Smarthack/gridpilot-ai/data/processed/weather.csv', index=False)

In [13]:
import json
seg_geo = w.groupby('span_id')[['latitude','longitude']].first()
AZIMUTH = {'S1': 40.0, 'S2': 95.0, 'S3': 150.0}   # SIMULAT, înlocuiește cu valorile de la P1
segs = [{'segment_id': str(k), 'latitude': r.latitude, 'longitude': r.longitude,
         'azimuth_deg': AZIMUTH[str(k)]} for k, r in seg_geo.iterrows()]
json.dump({'segments': segs}, open('config/line_segments.json', 'w'), indent=1)
print(segs)

[{'segment_id': 'S1', 'latitude': np.float64(44.38), 'longitude': np.float64(28.14), 'azimuth_deg': 40.0}, {'segment_id': 'S2', 'latitude': np.float64(44.46), 'longitude': np.float64(28.48), 'azimuth_deg': 95.0}, {'segment_id': 'S3', 'latitude': np.float64(44.55), 'longitude': np.float64(28.81), 'azimuth_deg': 150.0}]


In [10]:
w2 = pd.read_csv('Smarthack/gridpilot-ai/data/processed/weather.csv')
print(w2.shape)

(105165, 8)


In [14]:
!pip install -q linerate

In [19]:
import sys, logging
sys.path.insert(0, '.')
logging.basicConfig(level=logging.INFO)
from src import physics_engine as pe

cond = pe.load_conductor_config()
segs = pe.load_segments('config/line_segments.json')
for s in segs:
    print(s['segment_id'], pe.compute_static_rating(s, cond))

AttributeError: module 'src.physics_engine' has no attribute 'load_conductor_config'

In [20]:
!mkdir -p config data/processed
!cp /content/config/line_segments.json config/
!cp /content/data/processed/weather.csv data/processed/

cp: cannot stat '/content/data/processed/weather.csv': No such file or directory


In [18]:
%cd /content/Smarthack/gridpilot-ai

/content/Smarthack/gridpilot-ai


In [21]:
!ls data/processed config

config:
line_segments.json

data/processed:
dataset.csv  weather.csv


In [22]:
!ls config src

config:
line_segments.json

src:
agent.py	  decision_engine.py  physics_engine.py  report_generator.py
data_pipeline.py  ml_model.py	      __pycache__


In [23]:
!git pull origin main

remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 37 (delta 7), reused 35 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 6.50 MiB | 7.47 MiB/s, done.
From https://github.com/Fandrei19/Smarthack
 * branch            main       -> FETCH_HEAD
   1897616..a866770  main       -> origin/main
Updating 1897616..a866770
error: The following untracked working tree files would be overwritten by merge:
	gridpilot-ai/config/line_segments.json
	gridpilot-ai/data/processed/weather.csv
Please move or remove them before you merge.
Aborting


In [24]:
!pip install -q linerate numpy pandas scikit-learn joblib matplotlib seaborn pytest

In [25]:
import sys
import linerate
import sklearn
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("LineRate:", linerate.__file__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
LineRate: /usr/local/lib/python3.13/dist-packages/linerate/__init__.py
Scikit-learn: 1.6.1
Pandas: 2.2.3


In [31]:
import pandas as pd

df = pd.read_csv("data/processed/weather.csv")

print("Dimensiune:", df.shape)
print("\nColoane:")
print(df.columns.tolist())

print("\nPrimele 5 rânduri:")
display(df.head())

print("\nTipuri de date:")
print(df.dtypes)

print("\nValori lipsă:")
print(df.isnull().sum())

Dimensiune: (105165, 8)

Coloane:
['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_direction_deg', 'solar_radiation', 'source', 'data_quality']

Primele 5 rânduri:


,timestamp_utc,segment_id,air_temp_c,wind_speed_ms,wind_direction_deg,solar_radiation,source,data_quality
0,2022-01-01T06:00:00Z,S1,5.0,3.20,273,1,dataset.csv,unknown
1,2022-01-01T06:00:00Z,S2,4.9,3.77,281,1,dataset.csv,unknown
2,2022-01-01T06:00:00Z,S3,6.2,3.43,278,1,dataset.csv,unknown
3,2022-01-01T07:00:00Z,S1,5.3,3.26,274,26,dataset.csv,unknown
4,2022-01-01T07:00:00Z,S2,5.1,3.93,282,30,dataset.csv,unknown



Tipuri de date:
timestamp_utc          object
segment_id             object
air_temp_c            float64
wind_speed_ms         float64
wind_direction_deg      int64
solar_radiation         int64
source                 object
data_quality           object
dtype: object

Valori lipsă:
timestamp_utc         0
segment_id            0
air_temp_c            0
wind_speed_ms         0
wind_direction_deg    0
solar_radiation       0
source                0
data_quality          0
dtype: int64


In [29]:
import os

print("Director curent:", os.getcwd())
print("\nConținutul directorului:")
print(os.listdir("."))

Director curent: /content/Smarthack/gridpilot-ai

Conținutul directorului:
['data', 'app.py', 'config', 'requirements.txt', 'notebooks', 'evaluation.py', 'README.md', 'src', 'models']


In [33]:
import pandas as pd

weather = pd.read_csv("data/processed/weather.csv")

print("=== WEATHER.CSV ===")
print("Dimensiune:", weather.shape)
print("Coloane:", weather.columns.tolist())
display(weather.head())
print("\nValori lipsă:")
print(weather.isnull().sum())

# Verificăm și dataset.csv
DATASET_PATH = "data/processed/weather.csv".replace("weather.csv", "dataset.csv")
dataset = pd.read_csv(DATASET_PATH)

print("\n=== DATASET.CSV ===")
print("Dimensiune:", dataset.shape)
print("Coloane:", dataset.columns.tolist())
display(dataset.head())

=== WEATHER.CSV ===
Dimensiune: (105165, 8)
Coloane: ['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_direction_deg', 'solar_radiation', 'source', 'data_quality']


,timestamp_utc,segment_id,air_temp_c,wind_speed_ms,wind_direction_deg,solar_radiation,source,data_quality
0,2022-01-01T06:00:00Z,S1,5.0,3.20,273,1,dataset.csv,unknown
1,2022-01-01T06:00:00Z,S2,4.9,3.77,281,1,dataset.csv,unknown
2,2022-01-01T06:00:00Z,S3,6.2,3.43,278,1,dataset.csv,unknown
3,2022-01-01T07:00:00Z,S1,5.3,3.26,274,26,dataset.csv,unknown
4,2022-01-01T07:00:00Z,S2,5.1,3.93,282,30,dataset.csv,unknown



Valori lipsă:
timestamp_utc         0
segment_id            0
air_temp_c            0
wind_speed_ms         0
wind_direction_deg    0
solar_radiation       0
source                0
data_quality          0
dtype: int64

=== DATASET.CSV ===
Dimensiune: (105165, 35)
Coloane: ['timestamp_utc', 'span_id', 'target_timestamp_utc', 'air_temp_c', 'wind_speed_ms', 'wind_from_deg', 'solar_ghi_wm2', 'wind_dir_sin', 'wind_dir_cos', 'air_temp_c_lag_1h', 'wind_speed_ms_lag_1h', 'wind_dir_sin_lag_1h', 'wind_dir_cos_lag_1h', 'solar_ghi_wm2_lag_1h', 'air_temp_c_lag_3h', 'wind_speed_ms_lag_3h', 'wind_dir_sin_lag_3h', 'wind_dir_cos_lag_3h', 'solar_ghi_wm2_lag_3h', 'air_temp_c_lag_6h', 'wind_speed_ms_lag_6h', 'wind_dir_sin_lag_6h', 'wind_dir_cos_lag_6h', 'solar_ghi_wm2_lag_6h', 'hour_utc', 'month_utc', 'target_air_temp_c_1h', 'target_wind_speed_ms_1h', 'target_wind_from_deg_1h', 'target_solar_ghi_wm2_1h', 'latitude', 'longitude', 'era5_latitude', 'era5_longitude', 'split']


,timestamp_utc,span_id,target_timestamp_utc,air_temp_c,wind_speed_ms,wind_from_deg,solar_ghi_wm2,wind_dir_sin,wind_dir_cos,air_temp_c_lag_1h,...,month_utc,target_air_temp_c_1h,target_wind_speed_ms_1h,target_wind_from_deg_1h,target_solar_ghi_wm2_1h,latitude,longitude,era5_latitude,era5_longitude,split
0,2022-01-01T06:00:00Z,S1,2022-01-01T07:00:00Z,5.0,3.20,273,1,-0.998630,0.052336,5.0,...,1,5.3,3.26,274,26,44.38,28.14,44.50,28.25,train
1,2022-01-01T06:00:00Z,S2,2022-01-01T07:00:00Z,4.9,3.77,281,1,-0.981627,0.190809,5.0,...,1,5.1,3.93,282,30,44.46,28.48,44.50,28.50,train
2,2022-01-01T06:00:00Z,S3,2022-01-01T07:00:00Z,6.2,3.43,278,1,-0.990268,0.139173,6.3,...,1,6.2,3.88,277,34,44.55,28.81,44.75,28.75,train
3,2022-01-01T07:00:00Z,S1,2022-01-01T08:00:00Z,5.3,3.26,274,26,-0.997564,0.069756,5.0,...,1,6.0,3.26,275,83,44.38,28.14,44.50,28.25,train
4,2022-01-01T07:00:00Z,S2,2022-01-01T08:00:00Z,5.1,3.93,282,30,-0.978148,0.207912,4.9,...,1,6.0,4.06,284,88,44.46,28.48,44.50,28.50,train


In [34]:
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
CONFIG_DIR = os.path.join(PROJECT_DIR, "config")

print("Fișiere din config:\n")

for root, dirs, files in os.walk(CONFIG_DIR):
    for file in files:
        print(os.path.join(root, file))

Fișiere din config:

/content/Smarthack/gridpilot-ai/config/line_segments.json


In [35]:
import json

CONFIG_PATH = (
    "/content/Smarthack/gridpilot-ai/config/line_segments.json"
)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    line_config = json.load(f)

print(json.dumps(line_config, indent=4, ensure_ascii=False))

{
    "segments": [
        {
            "segment_id": "S1",
            "latitude": 44.38,
            "longitude": 28.14,
            "azimuth_deg": 40.0
        },
        {
            "segment_id": "S2",
            "latitude": 44.46,
            "longitude": 28.48,
            "azimuth_deg": 95.0
        },
        {
            "segment_id": "S3",
            "latitude": 44.55,
            "longitude": 28.81,
            "azimuth_deg": 150.0
        }
    ]
}


In [36]:
!pip install -q linerate numpy pandas

In [37]:
import linerate
print("LineRate instalat cu succes!")

LineRate instalat cu succes!


In [38]:

%%writefile /content/Smarthack/gridpilot-ai/src/physics_engine.py

import json
import numpy as np
import pandas as pd
import linerate


# ============================================================
# GRIDPILOT AI - SIMULATED PHYSICS ENGINE
# Model: CIGRE TB 601, via Statnett LineRate
# IMPORTANT: All conductor/span parameters are simulation
# assumptions, not PPC infrastructure measurements.
# ============================================================

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
CONFIG_PATH = f"{PROJECT_DIR}/config/line_segments.json"

MAX_CONDUCTOR_TEMPERATURE_C = 75.0
SPAN_LENGTH_M = 300.0
CONDUCTOR_ALTITUDE_M = 50.0
NUM_CONDUCTORS = 1

GROUND_ALBEDO = 0.15

# CIGRE example uses this clearness ratio.
# It is a provisional assumption, NOT derived from weather.csv.
CLEARNES_RATIO = 0.5


def load_segments():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    return {
        item["segment_id"]: item
        for item in config["segments"]
    }


def create_conductor():
    """Conductor parameters from the official LineRate CIGRE example."""

    return linerate.Conductor(
        core_diameter=10.4e-3,
        conductor_diameter=28.1e-3,
        outer_layer_strand_diameter=2.2e-3,
        emissivity=0.9,
        solar_absorptivity=0.9,
        temperature1=25,
        temperature2=75,
        resistance_at_temperature1=7.283e-5,
        resistance_at_temperature2=8.688e-5,
        aluminium_cross_section_area=float("nan"),
        constant_magnetic_effect=1,
        current_density_proportional_magnetic_effect=0,
        max_magnetic_core_relative_resistance_increase=1,
    )


def destination_point(lat, lon, bearing_deg, distance_m):
    """Calculate a point at a given bearing and distance."""

    earth_radius = 6371000.0

    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    bearing = np.radians(bearing_deg)
    angular_distance = distance_m / earth_radius

    lat2 = np.arcsin(
        np.sin(lat1) * np.cos(angular_distance)
        + np.cos(lat1) * np.sin(angular_distance) * np.cos(bearing)
    )

    lon2 = lon1 + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(lat1),
        np.cos(angular_distance) - np.sin(lat1) * np.sin(lat2)
    )

    return np.degrees(lat2), np.degrees(lon2)


def create_span(segment, conductor):
    """Create a simulated 300 m span centered on segment coordinates."""

    lat = segment["latitude"]
    lon = segment["longitude"]
    azimuth = segment["azimuth_deg"]

    start_lat, start_lon = destination_point(
        lat, lon, azimuth + 180, SPAN_LENGTH_M / 2
    )

    end_lat, end_lon = destination_point(
        lat, lon, azimuth, SPAN_LENGTH_M / 2
    )

    start_tower = linerate.Tower(
        latitude=start_lat,
        longitude=start_lon,
        altitude=CONDUCTOR_ALTITUDE_M,
    )

    end_tower = linerate.Tower(
        latitude=end_lat,
        longitude=end_lon,
        altitude=CONDUCTOR_ALTITUDE_M,
    )

    return linerate.Span(
        conductor=conductor,
        start_tower=start_tower,
        end_tower=end_tower,
        num_conductors=NUM_CONDUCTORS,
    )


def calculate_ampacity(row, segment):
    """Calculate steady-state ampacity for one weather observation."""

    conductor = create_conductor()
    span = create_span(segment, conductor)

    weather = linerate.Weather(
        air_temperature=float(row["air_temp_c"]),
        wind_direction=np.radians(
            float(row["wind_direction_deg"])
        ),
        wind_speed=float(row["wind_speed_ms"]),
        ground_albedo=GROUND_ALBEDO,
        clearness_ratio=CLEARNES_RATIO,
    )

    timestamp = pd.Timestamp(row["timestamp_utc"])
    timestamp = timestamp.tz_convert(None).to_datetime64()

    model = linerate.Cigre601(
        span,
        weather,
        timestamp,
    )

    ampacity = model.compute_steady_state_ampacity(
        max_conductor_temperature=MAX_CONDUCTOR_TEMPERATURE_C,
        min_ampacity=0,
        max_ampacity=3000,
        tolerance=1.0,
    )

    return float(ampacity)


def generate_training_dataset(weather_df, limit=None):
    """Generate physical ampacity labels for weather observations."""

    segments = load_segments()

    required = {
        "timestamp_utc",
        "segment_id",
        "air_temp_c",
        "wind_speed_ms",
        "wind_direction_deg",
    }

    missing = required - set(weather_df.columns)

    if missing:
        raise ValueError(f"Missing weather columns: {missing}")

    data = weather_df.copy()

    if limit is not None:
        data = data.head(limit).copy()

    results = []

    for index, row in data.iterrows():
        segment_id = row["segment_id"]

        if segment_id not in segments:
            raise ValueError(f"Unknown segment: {segment_id}")

        ampacity = calculate_ampacity(
            row,
            segments[segment_id],
        )

        results.append(ampacity)

        if len(results) % 100 == 0:
            print(f"Processed {len(results)} observations")

    data["target_ampacity_a"] = results

    return data


Overwriting /content/Smarthack/gridpilot-ai/src/physics_engine.py


In [39]:
import sys
sys.path.insert(
    0,
    "/content/Smarthack/gridpilot-ai/src"
)

from physics_engine import generate_training_dataset

test_data = generate_training_dataset(
    weather,
    limit=30
)

display(test_data.head(10))

print("\nStatistici ampacitate:")
print(test_data["target_ampacity_a"].describe())

,timestamp_utc,segment_id,air_temp_c,wind_speed_ms,wind_direction_deg,solar_radiation,source,data_quality,target_ampacity_a
0,2022-01-01T06:00:00Z,S1,5.0,3.20,273,1,dataset.csv,unknown,1567.749023
1,2022-01-01T06:00:00Z,S2,4.9,3.77,281,1,dataset.csv,unknown,1235.229492
2,2022-01-01T06:00:00Z,S3,6.2,3.43,278,1,dataset.csv,unknown,1556.030273
3,2022-01-01T07:00:00Z,S1,5.3,3.26,274,26,dataset.csv,unknown,1547.973633
4,2022-01-01T07:00:00Z,S2,5.1,3.93,282,30,dataset.csv,unknown,1221.313477
5,2022-01-01T07:00:00Z,S3,6.2,3.88,277,34,dataset.csv,unknown,1550.903320
6,2022-01-01T08:00:00Z,S1,6.0,3.26,275,83,dataset.csv,unknown,1534.790039
7,2022-01-01T08:00:00Z,S2,6.0,4.06,284,88,dataset.csv,unknown,1216.918945
8,2022-01-01T08:00:00Z,S3,7.5,4.24,282,95,dataset.csv,unknown,1509.155273
9,2022-01-01T09:00:00Z,S1,7.2,3.05,272,187,dataset.csv,unknown,1506.958008



Statistici ampacitate:
count      30.000000
mean     1407.543945
std       142.069264
min      1145.874023
25%      1284.667969
50%      1426.025391
75%      1535.888672
max      1583.129883
Name: target_ampacity_a, dtype: float64


In [41]:
import pandas as pd

TRAINING_PATH = "data/processed/training_dataset.csv"

training = pd.read_csv(TRAINING_PATH)
weather = pd.read_csv(WEATHER_PATH)

print("Training dataset:", training.shape)
print("Weather dataset:", weather.shape)

print("\nObservații per segment:")
print(training.groupby("segment_id").size())

print("\nTimestamp-uri distincte:")
print(training["timestamp_utc"].nunique())

print("\nValori lipsă:")
print(training.isnull().sum())

print("\nDistribuția ampacității:")
print(training["target_ampacity_a"].describe())

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/training_dataset.csv'

In [42]:
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if "training" in file.lower() or "weather" in file.lower():
            print(os.path.join(root, file))

/content/Smarthack/gridpilot-ai/data/processed/weather.csv
/content/Smarthack/gridpilot-ai/notebooks/02_model_training.ipynb


In [43]:
import json

NOTEBOOK_PATH = (
    "/content/Smarthack/gridpilot-ai/notebooks/02_model_training.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        print(f"\n{'='*60}")
        print(f"CELL {i}")
        print(code[:3000])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [44]:
import os

NOTEBOOK_PATH = "/content/Smarthack/gridpilot-ai/notebooks/02_model_training.ipynb"

print("Există:", os.path.exists(NOTEBOOK_PATH))
print("Dimensiune:", os.path.getsize(NOTEBOOK_PATH), "bytes")

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    print("Primele 500 caractere:")
    print(repr(f.read(500)))

Există: True
Dimensiune: 0 bytes
Primele 500 caractere:
''


In [45]:
import os

NOTEBOOK_DIR = "/content/Smarthack/gridpilot-ai/notebooks"

for file in os.listdir(NOTEBOOK_DIR):
    path = os.path.join(NOTEBOOK_DIR, file)
    print(file, "->", os.path.getsize(path), "bytes")

01_data_exploration.ipynb -> 0 bytes
02_model_training.ipynb -> 0 bytes


In [46]:
import os
import importlib.util

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

print("=== STRUCTURA PROIECTULUI ===")

for folder in ["src", "data", "models", "tests", "config"]:
    path = os.path.join(PROJECT_DIR, folder)

    print(f"\n{folder}:")

    if os.path.exists(path):
        for item in os.listdir(path):
            print("  ", item)
    else:
        print("  Folder inexistent")

print("\n=== LINERATE ===")
print("Instalat:", importlib.util.find_spec("linerate") is not None)

=== STRUCTURA PROIECTULUI ===

src:
   agent.py
   __pycache__
   decision_engine.py
   report_generator.py
   ml_model.py
   data_pipeline.py
   physics_engine.py

data:
   raw
   processed

models:
   ampacity_model.pkl

tests:
  Folder inexistent

config:
   line_segments.json

=== LINERATE ===
Instalat: True


In [47]:
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

files_to_inspect = [
    "src/physics_engine.py",
    "src/ml_model.py",
    "src/data_pipeline.py",
]

for relative_path in files_to_inspect:
    path = os.path.join(PROJECT_DIR, relative_path)

    print("\n" + "=" * 80)
    print(relative_path)
    print("=" * 80)

    with open(path, "r", encoding="utf-8") as f:
        print(f.read())


src/physics_engine.py

import json
import numpy as np
import pandas as pd
import linerate


# ============================================================
# GRIDPILOT AI - SIMULATED PHYSICS ENGINE
# Model: CIGRE TB 601, via Statnett LineRate
# IMPORTANT: All conductor/span parameters are simulation
# assumptions, not PPC infrastructure measurements.
# ============================================================

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
CONFIG_PATH = f"{PROJECT_DIR}/config/line_segments.json"

MAX_CONDUCTOR_TEMPERATURE_C = 75.0
SPAN_LENGTH_M = 300.0
CONDUCTOR_ALTITUDE_M = 50.0
NUM_CONDUCTORS = 1

GROUND_ALBEDO = 0.15

# CIGRE example uses this clearness ratio.
# It is a provisional assumption, NOT derived from weather.csv.
CLEARNES_RATIO = 0.5


def load_segments():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    return {
        item["segment_id"]: item
        for item in config["segments"]
    }


def create_conducto

In [48]:
import os
import pickle

MODEL_PATH = os.path.join(
    PROJECT_DIR, "models", "ampacity_model.pkl"
)

print("Model existent:", os.path.exists(MODEL_PATH))
print("Dimensiune:", os.path.getsize(MODEL_PATH), "bytes")

if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)

    print("Tip model:", type(model))
    print("Detalii:", model)

Model existent: True
Dimensiune: 0 bytes


EOFError: Ran out of input

In [49]:
import os

MODEL_PATH = "/content/Smarthack/gridpilot-ai/models/ampacity_model.pkl"

print("Dimensiune:", os.path.getsize(MODEL_PATH), "bytes")

Dimensiune: 0 bytes


In [50]:
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

for filename in ["physics_engine.py", "ml_model.py"]:
    path = os.path.join(PROJECT_DIR, "src", filename)

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    print("Dimensiune:", len(content), "caractere")
    print(content)



physics_engine.py
Dimensiune: 5035 caractere

import json
import numpy as np
import pandas as pd
import linerate


# ============================================================
# GRIDPILOT AI - SIMULATED PHYSICS ENGINE
# Model: CIGRE TB 601, via Statnett LineRate
# IMPORTANT: All conductor/span parameters are simulation
# assumptions, not PPC infrastructure measurements.
# ============================================================

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
CONFIG_PATH = f"{PROJECT_DIR}/config/line_segments.json"

MAX_CONDUCTOR_TEMPERATURE_C = 75.0
SPAN_LENGTH_M = 300.0
CONDUCTOR_ALTITUDE_M = 50.0
NUM_CONDUCTORS = 1

GROUND_ALBEDO = 0.15

# CIGRE example uses this clearness ratio.
# It is a provisional assumption, NOT derived from weather.csv.
CLEARNES_RATIO = 0.5


def load_segments():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    return {
        item["segment_id"]: item
        for item in config["segments"]
    

In [51]:
import os

model_path = "/content/Smarthack/gridpilot-ai/models/ampacity_model.pkl"

print("Există:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Dimensiune:", os.path.getsize(model_path), "bytes")

Există: True
Dimensiune: 0 bytes


In [52]:
import os
import pandas as pd

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

weather_path = f"{PROJECT_DIR}/data/processed/weather.csv"
dataset_path = f"{PROJECT_DIR}/data/processed/dataset.csv"
model_path = f"{PROJECT_DIR}/models/ampacity_model.pkl"

weather = pd.read_csv(weather_path)
dataset = pd.read_csv(dataset_path)

print("WEATHER.CSV")
print("Shape:", weather.shape)
print("Coloane:", weather.columns.tolist())
print(weather[[
    "timestamp_utc",
    "segment_id",
    "air_temp_c",
    "wind_speed_ms",
    "wind_direction_deg",
    "solar_radiation"
]].head())

print("\nDATASET.CSV")
print("Shape:", dataset.shape)
print("Coloane:", dataset.columns.tolist())

print("\nAMPACITY MODEL")
print("Există:", os.path.exists(model_path))
if os.path.exists(model_path):
    print("Dimensiune:", os.path.getsize(model_path), "bytes")

print("\nNUMĂR OBSERVAȚII PE SEGMENT")
print(weather["segment_id"].value_counts())

print("\nDIRECȚIA VÂNTULUI")
print(weather["wind_direction_deg"].describe())

print("\nRADIAȚIA SOLARĂ")
print(weather["solar_radiation"].describe())

WEATHER.CSV
Shape: (105165, 8)
Coloane: ['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_direction_deg', 'solar_radiation', 'source', 'data_quality']
          timestamp_utc segment_id  air_temp_c  wind_speed_ms  \
0  2022-01-01T06:00:00Z         S1         5.0           3.20   
1  2022-01-01T06:00:00Z         S2         4.9           3.77   
2  2022-01-01T06:00:00Z         S3         6.2           3.43   
3  2022-01-01T07:00:00Z         S1         5.3           3.26   
4  2022-01-01T07:00:00Z         S2         5.1           3.93   

   wind_direction_deg  solar_radiation  
0                 273                1  
1                 281                1  
2                 278                1  
3                 274               26  
4                 282               30  

DATASET.CSV
Shape: (105165, 35)
Coloane: ['timestamp_utc', 'span_id', 'target_timestamp_utc', 'air_temp_c', 'wind_speed_ms', 'wind_from_deg', 'solar_ghi_wm2', 'wind_dir_sin', 'wind_dir_cos', '

In [53]:
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

# Verificăm sursele și calitatea datelor
weather = pd.read_csv(
    f"{PROJECT_DIR}/data/processed/weather.csv"
)

print("SURSE:")
print(weather["source"].value_counts(dropna=False))

print("\nCALITATE:")
print(weather["data_quality"].value_counts(dropna=False))

print("\nRADIAȚIE SOLARĂ - VALORI DISTINCTE")
print(sorted(weather["solar_radiation"].dropna().unique())[:30])

# Verificăm implementarea pipeline-ului
pipeline_path = f"{PROJECT_DIR}/src/data_pipeline.py"

print("\n" + "=" * 60)
print("DATA_PIPELINE.PY")
print("=" * 60)

with open(pipeline_path, "r", encoding="utf-8") as f:
    print(f.read())

SURSE:
source
dataset.csv    105165
Name: count, dtype: int64

CALITATE:
data_quality
unknown    105165
Name: count, dtype: int64

RADIAȚIE SOLARĂ - VALORI DISTINCTE
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]

DATA_PIPELINE.PY
"""Build an hourly dataset for one-hour-ahead DLR weather prediction.

Source: Open-Meteo Historical Weather API, ERA5 reanalysis (CC BY 4.0).
https://open-meteo.com/en/docs/historical-weather-api
Demo locations represent synthetic spans, not monitored power lines.
Requires pandas. Run: python data_pipeline.py
"""

import argparse
import json
import time
from pathlib import Path
from urllib.error impor

In [54]:
import inspect
import linerate

print("LineRate version:", getattr(linerate, "__version__", "unknown"))

print("\nWeatherWithSolarRadiation:")
print(inspect.signature(linerate.WeatherWithSolarRadiation))

print("\nCigre601WithSolarRadiation:")
print(inspect.signature(linerate.Cigre601WithSolarRadiation))

print("\nWeather:")
print(inspect.signature(linerate.Weather))

print("\nAmpacity method:")
print(inspect.signature(
    linerate.Cigre601WithSolarRadiation.compute_steady_state_ampacity
))

LineRate version: 5.0.0

WeatherWithSolarRadiation:
(air_temperature: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], '°C'], wind_direction: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'rad'], wind_speed: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'm/s'], ground_albedo: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], ''], diffuse_radiation_intensity: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'W/m²'], direct_radiation_intensity: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'W/m²']) -> None

Cigre601WithSolarRadiation:


AttributeError: module 'linerate' has no attribute 'Cigre601WithSolarRadiation'

In [55]:
import inspect
import linerate

from linerate.models.cigre601 import Cigre601WithSolarRadiation
from linerate.types import WeatherWithSolarRadiation

print("LineRate:", linerate.__version__)

print("\nCigre601WithSolarRadiation:")
print(inspect.signature(Cigre601WithSolarRadiation))

print("\nWeatherWithSolarRadiation:")
print(inspect.signature(WeatherWithSolarRadiation))

print("\nAmpacity method:")
print(inspect.signature(
    Cigre601WithSolarRadiation.compute_steady_state_ampacity
))

LineRate: 5.0.0

Cigre601WithSolarRadiation:
(span: linerate.types.Span, weather: linerate.types.WeatherWithSolarRadiation, time: Union[numpy.datetime64, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.datetime64]]], max_reynolds_number: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], ''] = 4000.0)

WeatherWithSolarRadiation:
(air_temperature: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], '°C'], wind_direction: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'rad'], wind_speed: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], 'm/s'], ground_albedo: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating]]], ''], diffuse_radiation_intensity: Annotated[Union[float, numpy.floating, numpy.ndarray[tuple[Any, ...], numpy.dtype[numpy.floating

In [56]:
from urllib.parse import urlencode
from urllib.request import urlopen
import json

params = {
    "latitude": 44.38,
    "longitude": 28.14,
    "start_date": "2025-06-01",
    "end_date": "2025-06-02",
    "hourly": (
        "shortwave_radiation,"
        "direct_normal_irradiance,"
        "diffuse_radiation"
    ),
    "models": "era5",
    "timezone": "GMT",
}

url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    + urlencode(params)
)

with urlopen(url, timeout=35) as response:
    payload = json.load(response)

print("Unități:")
print(payload["hourly_units"])

print("\nPrimele 10 observații:")
for i in range(10):
    print(
        payload["hourly"]["time"][i],
        "GHI:", payload["hourly"]["shortwave_radiation"][i],
        "DNI:", payload["hourly"]["direct_normal_irradiance"][i],
        "DHI:", payload["hourly"]["diffuse_radiation"][i],
    )

Unități:
{'time': 'iso8601', 'shortwave_radiation': 'W/m²', 'direct_normal_irradiance': 'W/m²', 'diffuse_radiation': 'W/m²'}

Primele 10 observații:
2025-06-01T00:00 GHI: 0.0 DNI: 0.0 DHI: 0.0
2025-06-01T01:00 GHI: 0.0 DNI: 0.0 DHI: 0.0
2025-06-01T02:00 GHI: 0.0 DNI: 0.0 DHI: 0.0
2025-06-01T03:00 GHI: 6.0 DNI: 18.1 DHI: 5.0
2025-06-01T04:00 GHI: 78.0 DNI: 165.0 DHI: 51.0
2025-06-01T05:00 GHI: 217.0 DNI: 300.9 DHI: 116.0
2025-06-01T06:00 GHI: 378.0 DNI: 483.3 DHI: 135.0
2025-06-01T07:00 GHI: 567.0 DNI: 605.8 DHI: 171.0
2025-06-01T08:00 GHI: 724.0 DNI: 717.4 DHI: 166.0
2025-06-01T09:00 GHI: 646.0 DNI: 504.0 DHI: 209.0


In [57]:
import shutil
import os

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

dataset_path = f"{PROJECT_DIR}/data/processed/dataset.csv"
backup_path = f"{PROJECT_DIR}/data/processed/dataset_before_solar.csv"

if os.path.exists(dataset_path):
    shutil.copy2(dataset_path, backup_path)
    print("Backup creat:", backup_path)

Backup creat: /content/Smarthack/gridpilot-ai/data/processed/dataset_before_solar.csv


In [58]:
!python /content/Smarthack/gridpilot-ai/src/data_pipeline.py \
    --output /content/Smarthack/gridpilot-ai/data/processed/dataset.csv

S1: 35,055 rows kept of 35,064 source hours (ERA5 grid: 44.5, 28.25)
S2: 35,055 rows kept of 35,064 source hours (ERA5 grid: 44.5, 28.5)
S3: 35,055 rows kept of 35,064 source hours (ERA5 grid: 44.75, 28.75)
span_id  split     
S1       test           8759
         train         17513
         validation     8783
S2       test           8759
         train         17513
         validation     8783
S3       test           8759
         train         17513
         validation     8783
Saved 105,165 rows to /content/Smarthack/gridpilot-ai/data/processed/dataset.csv


In [59]:
import pandas as pd

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

df = pd.read_csv(
    f"{PROJECT_DIR}/data/processed/dataset.csv"
)

print("Shape:", df.shape)

solar_columns = [
    "solar_ghi_wm2",
    "solar_direct_normal_wm2",
    "solar_diffuse_wm2",
]

print("\nColoane solare:")
print(df[solar_columns].head(10))

print("\nStatistici:")
print(df[solar_columns].describe())

print("\nSplit:")
print(df["split"].value_counts())

Shape: (105165, 39)

Coloane solare:
   solar_ghi_wm2  solar_direct_normal_wm2  solar_diffuse_wm2
0              1                      0.0                  1
1              1                      0.0                  1
2              1                      0.0                  1
3             26                      0.0                 26
4             30                      9.7                 29
5             34                     19.5                 32
6             83                     17.6                 79
7             88                     17.5                 84
8             95                     74.8                 78
9            187                    172.4                132

Statistici:
       solar_ghi_wm2  solar_direct_normal_wm2  solar_diffuse_wm2
count  105165.000000            105165.000000      105165.000000
mean      167.875624               199.380237          57.341654
std       242.124463               272.220824          74.532323
min         0.00000

In [60]:
from pathlib import Path

PROJECT_DIR = Path("/content/Smarthack/gridpilot-ai")

for path in PROJECT_DIR.rglob("*.py"):
    if any(part.startswith(".") for part in path.parts):
        continue

    content = path.read_text(encoding="utf-8", errors="ignore")

    if "weather.csv" in content or "solar_radiation" in content:
        print("\n" + "=" * 60)
        print(path)
        print("=" * 60)

        for i, line in enumerate(content.splitlines(), 1):
            if "weather.csv" in line or "solar_radiation" in line:
                print(f"{i}: {line}")


/content/Smarthack/gridpilot-ai/src/physics_engine.py
26: # It is a provisional assumption, NOT derived from weather.csv.


In [61]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/Smarthack/gridpilot-ai")

dataset_path = PROJECT_DIR / "data/processed/dataset.csv"
weather_path = PROJECT_DIR / "data/processed/weather.csv"

df = pd.read_csv(dataset_path)

weather = df[[
    "timestamp_utc",
    "span_id",
    "air_temp_c",
    "wind_speed_ms",
    "wind_from_deg",
    "solar_ghi_wm2",
    "solar_direct_normal_wm2",
    "solar_diffuse_wm2",
]].copy()

weather = weather.rename(columns={
    "span_id": "segment_id",
    "wind_from_deg": "wind_direction_deg",
    "solar_ghi_wm2": "solar_radiation",
})

weather["source"] = "dataset.csv"
weather["data_quality"] = "unknown"

weather = weather.sort_values(
    ["timestamp_utc", "segment_id"]
).reset_index(drop=True)

weather.to_csv(weather_path, index=False)

print("Salvat:", weather_path)
print("Shape:", weather.shape)
print("Coloane:", weather.columns.tolist())
print(weather.head())

Salvat: /content/Smarthack/gridpilot-ai/data/processed/weather.csv
Shape: (105165, 10)
Coloane: ['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_direction_deg', 'solar_radiation', 'solar_direct_normal_wm2', 'solar_diffuse_wm2', 'source', 'data_quality']
          timestamp_utc segment_id  air_temp_c  wind_speed_ms  \
0  2022-01-01T06:00:00Z         S1         5.0           3.20   
1  2022-01-01T06:00:00Z         S2         4.9           3.77   
2  2022-01-01T06:00:00Z         S3         6.2           3.43   
3  2022-01-01T07:00:00Z         S1         5.3           3.26   
4  2022-01-01T07:00:00Z         S2         5.1           3.93   

   wind_direction_deg  solar_radiation  solar_direct_normal_wm2  \
0                 273                1                      0.0   
1                 281                1                      0.0   
2                 278                1                      0.0   
3                 274               26                      0.0   


In [62]:
weather = pd.read_csv(weather_path)

required_columns = [
    "timestamp_utc",
    "segment_id",
    "air_temp_c",
    "wind_speed_ms",
    "wind_direction_deg",
    "solar_radiation",
    "solar_direct_normal_wm2",
    "solar_diffuse_wm2",
    "source",
    "data_quality",
]

assert all(col in weather.columns for col in required_columns)
assert len(weather) == 105165
assert not weather[required_columns].isnull().any().any()

print("Validare reușită!")
print(weather.groupby("segment_id").size())
print(weather[[
    "solar_radiation",
    "solar_direct_normal_wm2",
    "solar_diffuse_wm2"
]].describe())

Validare reușită!
segment_id
S1    35055
S2    35055
S3    35055
dtype: int64
       solar_radiation  solar_direct_normal_wm2  solar_diffuse_wm2
count    105165.000000            105165.000000      105165.000000
mean        167.875624               199.380237          57.341654
std         242.124463               272.220824          74.532323
min           0.000000                 0.000000           0.000000
25%           0.000000                 0.000000           0.000000
50%           8.000000                 0.000000           6.000000
75%         289.000000               400.700000         109.000000
max         939.000000               904.900000         431.000000


In [63]:
import sys
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
sys.path.insert(0, PROJECT_DIR)

import linerate
from linerate.models.cigre601 import Cigre601WithSolarRadiation
from linerate.types import WeatherWithSolarRadiation

from src.physics_engine import (
    create_conductor,
    create_span,
    load_segments,
    MAX_CONDUCTOR_TEMPERATURE_C,
    GROUND_ALBEDO,
)

weather_df = pd.read_csv(
    f"{PROJECT_DIR}/data/processed/weather.csv"
)

segments = load_segments()
conductor = create_conductor()

# Construim o singură dată fiecare span.
spans = {
    segment_id: create_span(segment, conductor)
    for segment_id, segment in segments.items()
}

results = []

for segment_id in ["S1", "S2", "S3"]:
    sample = weather_df[
        weather_df["segment_id"] == segment_id
    ].head(2)

    for _, row in sample.iterrows():
        timestamp = pd.Timestamp(row["timestamp_utc"])
        timestamp = timestamp.tz_convert(None).to_datetime64()

        weather = WeatherWithSolarRadiation(
            air_temperature=float(row["air_temp_c"]),
            wind_direction=np.radians(
                float(row["wind_direction_deg"])
            ),
            wind_speed=float(row["wind_speed_ms"]),
            ground_albedo=GROUND_ALBEDO,
            diffuse_radiation_intensity=float(
                row["solar_diffuse_wm2"]
            ),
            direct_radiation_intensity=float(
                row["solar_direct_normal_wm2"]
            ),
        )

        model = Cigre601WithSolarRadiation(
            spans[segment_id],
            weather,
            timestamp,
        )

        ampacity = model.compute_steady_state_ampacity(
            max_conductor_temperature=MAX_CONDUCTOR_TEMPERATURE_C,
            min_ampacity=0,
            max_ampacity=3000,
            tolerance=1.0,
        )

        results.append({
            "segment_id": segment_id,
            "timestamp_utc": row["timestamp_utc"],
            "air_temp_c": row["air_temp_c"],
            "wind_speed_ms": row["wind_speed_ms"],
            "solar_direct_normal_wm2": row["solar_direct_normal_wm2"],
            "solar_diffuse_wm2": row["solar_diffuse_wm2"],
            "ampacity_a": float(ampacity),
        })

test_results = pd.DataFrame(results)

print("LineRate:", linerate.__version__)
print(test_results.to_string(index=False))

assert np.isfinite(test_results["ampacity_a"]).all()
assert (test_results["ampacity_a"] > 0).all()

print("\nTest trecut: toate cele 6 rezultate sunt finite și pozitive.")

ImportError: cannot import name 'create_conductor' from 'src.physics_engine' (/content/Smarthack/gridpilot-ai/src/physics_engine.py)

In [64]:
import importlib
import src.physics_engine as pe

importlib.reload(pe)

print("Fișier:", pe.__file__)
print("create_conductor:", hasattr(pe, "create_conductor"))
print("create_span:", hasattr(pe, "create_span"))
print("load_segments:", hasattr(pe, "load_segments"))

print("\nFuncții disponibile:")
print([x for x in dir(pe) if not x.startswith("_")])

Fișier: /content/Smarthack/gridpilot-ai/src/physics_engine.py
create_conductor: True
create_span: True
load_segments: True

Funcții disponibile:
['CLEARNES_RATIO', 'CONDUCTOR_ALTITUDE_M', 'CONFIG_PATH', 'GROUND_ALBEDO', 'MAX_CONDUCTOR_TEMPERATURE_C', 'NUM_CONDUCTORS', 'PROJECT_DIR', 'SPAN_LENGTH_M', 'calculate_ampacity', 'create_conductor', 'create_span', 'destination_point', 'generate_training_dataset', 'json', 'linerate', 'load_segments', 'np', 'pd']


In [65]:
import numpy as np
import pandas as pd

from linerate.models.cigre601 import Cigre601WithSolarRadiation
from linerate.types import WeatherWithSolarRadiation

# Folosim modulul deja reîncărcat
create_conductor = pe.create_conductor
create_span = pe.create_span
load_segments = pe.load_segments

MAX_CONDUCTOR_TEMPERATURE_C = pe.MAX_CONDUCTOR_TEMPERATURE_C
GROUND_ALBEDO = pe.GROUND_ALBEDO

PROJECT_DIR = "/content/Smarthack/gridpilot-ai"

weather_df = pd.read_csv(
    f"{PROJECT_DIR}/data/processed/weather.csv"
)

segments = load_segments()
conductor = create_conductor()

# Construim fiecare span o singură dată
spans = {
    segment_id: create_span(segment, conductor)
    for segment_id, segment in segments.items()
}

results = []

for segment_id in ["S1", "S2", "S3"]:

    sample = weather_df[
        weather_df["segment_id"] == segment_id
    ].head(2)

    for _, row in sample.iterrows():

        timestamp = pd.Timestamp(row["timestamp_utc"])

        if timestamp.tzinfo is not None:
            timestamp = timestamp.tz_convert(None)

        timestamp = timestamp.to_datetime64()

        weather = WeatherWithSolarRadiation(
            air_temperature=float(row["air_temp_c"]),
            wind_direction=np.radians(
                float(row["wind_direction_deg"])
            ),
            wind_speed=float(row["wind_speed_ms"]),
            ground_albedo=GROUND_ALBEDO,
            diffuse_radiation_intensity=float(
                row["solar_diffuse_wm2"]
            ),
            direct_radiation_intensity=float(
                row["solar_direct_normal_wm2"]
            ),
        )

        model = Cigre601WithSolarRadiation(
            spans[segment_id],
            weather,
            timestamp
        )

        ampacity = model.compute_steady_state_ampacity(
            max_conductor_temperature=MAX_CONDUCTOR_TEMPERATURE_C,
            min_ampacity=0,
            max_ampacity=3000,
            tolerance=1.0
        )

        results.append({
            "segment_id": segment_id,
            "timestamp_utc": row["timestamp_utc"],
            "air_temp_c": row["air_temp_c"],
            "wind_speed_ms": row["wind_speed_ms"],
            "solar_direct_normal_wm2": row["solar_direct_normal_wm2"],
            "solar_diffuse_wm2": row["solar_diffuse_wm2"],
            "ampacity_a": float(ampacity)
        })

test_results = pd.DataFrame(results)

print(test_results.to_string(index=False))

assert np.isfinite(test_results["ampacity_a"]).all()
assert (test_results["ampacity_a"] > 0).all()

print("\nTest trecut: toate cele 6 valori de ampacitate sunt finite și pozitive.")

segment_id        timestamp_utc  air_temp_c  wind_speed_ms  solar_direct_normal_wm2  solar_diffuse_wm2  ampacity_a
        S1 2022-01-01T06:00:00Z         5.0           3.20                      0.0                  1 1577.270508
        S1 2022-01-01T07:00:00Z         5.3           3.26                      0.0                 26 1575.073242
        S2 2022-01-01T06:00:00Z         4.9           3.77                      0.0                  1 1243.286133
        S2 2022-01-01T07:00:00Z         5.1           3.93                      9.7                 29 1246.215820
        S3 2022-01-01T06:00:00Z         6.2           3.43                      0.0                  1 1561.157227
        S3 2022-01-01T07:00:00Z         6.2           3.88                     19.5                 32 1561.157227

Test trecut: toate cele 6 valori de ampacitate sunt finite și pozitive.


In [66]:

from pathlib import Path

PROJECT_DIR = Path("/content/Smarthack/gridpilot-ai")
PHYSICS_FILE = PROJECT_DIR / "src" / "physics_engine.py"

code = r'''
import json
import os
import numpy as np
import pandas as pd

from linerate.models.cigre601 import Cigre601WithSolarRadiation
from linerate.types import WeatherWithSolarRadiation


PROJECT_DIR = "/content/Smarthack/gridpilot-ai"
CONFIG_PATH = f"{PROJECT_DIR}/config/line_segments.json"

MAX_CONDUCTOR_TEMPERATURE_C = 75.0
SPAN_LENGTH_M = 300.0
CONDUCTOR_ALTITUDE_M = 50.0
NUM_CONDUCTORS = 1
GROUND_ALBEDO = 0.15


def load_segments():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    return {
        item["segment_id"]: item
        for item in config["segments"]
    }


def create_conductor():
    import linerate

    return linerate.Conductor(
        core_diameter=10.4e-3,
        conductor_diameter=28.1e-3,
        outer_layer_strand_diameter=2.2e-3,
        emissivity=0.9,
        solar_absorptivity=0.9,
        temperature1=25,
        temperature2=75,
        resistance_at_temperature1=7.283e-5,
        resistance_at_temperature2=8.688e-5,
        aluminium_cross_section_area=float("nan"),
        constant_magnetic_effect=1,
        current_density_proportional_magnetic_effect=0,
        max_magnetic_core_relative_resistance_increase=1,
    )


def destination_point(lat, lon, bearing_deg, distance_m):
    earth_radius = 6371000.0

    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    bearing = np.radians(bearing_deg)

    angular_distance = distance_m / earth_radius

    lat2 = np.arcsin(
        np.sin(lat1) * np.cos(angular_distance)
        + np.cos(lat1) * np.sin(angular_distance)
        * np.cos(bearing)
    )

    lon2 = lon1 + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(lat1),
        np.cos(angular_distance) - np.sin(lat1) * np.sin(lat2),
    )

    return np.degrees(lat2), np.degrees(lon2)


def create_span(segment, conductor):
    lat = segment["latitude"]
    lon = segment["longitude"]
    azimuth = segment["azimuth_deg"]

    start_lat, start_lon = destination_point(
        lat, lon, azimuth + 180, SPAN_LENGTH_M / 2
    )

    end_lat, end_lon = destination_point(
        lat, lon, azimuth, SPAN_LENGTH_M / 2
    )

    import linerate

    start_tower = linerate.Tower(
        latitude=start_lat,
        longitude=start_lon,
        altitude=CONDUCTOR_ALTITUDE_M,
    )

    end_tower = linerate.Tower(
        latitude=end_lat,
        longitude=end_lon,
        altitude=CONDUCTOR_ALTITUDE_M,
    )

    return linerate.Span(
        conductor=conductor,
        start_tower=start_tower,
        end_tower=end_tower,
        num_conductors=NUM_CONDUCTORS,
    )


def calculate_ampacity(row, span):
    timestamp = pd.Timestamp(row["timestamp_utc"])

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert(None)

    weather = WeatherWithSolarRadiation(
        air_temperature=float(row["air_temp_c"]),
        wind_direction=np.radians(
            float(row["wind_direction_deg"])
        ),
        wind_speed=float(row["wind_speed_ms"]),
        ground_albedo=GROUND_ALBEDO,
        diffuse_radiation_intensity=float(
            row["solar_diffuse_wm2"]
        ),
        direct_radiation_intensity=float(
            row["solar_direct_normal_wm2"]
        ),
    )

    model = Cigre601WithSolarRadiation(
        span,
        weather,
        timestamp.to_datetime64(),
    )

    ampacity = model.compute_steady_state_ampacity(
        max_conductor_temperature=MAX_CONDUCTOR_TEMPERATURE_C,
        min_ampacity=0,
        max_ampacity=3000,
        tolerance=1.0,
    )

    ampacity = float(ampacity)

    if not np.isfinite(ampacity) or ampacity <= 0:
        raise ValueError(
            f"Ampacitate invalidă: {ampacity}, "
            f"segment={row['segment_id']}, "
            f"timestamp={row['timestamp_utc']}"
        )

    return ampacity


def generate_training_dataset(
    weather_df,
    output_path=None,
    limit=None,
    checkpoint_every=2000,
    resume=True,
):
    """
    Generează etichetele de ampacitate folosind CIGRE TB 601
    cu radiație solară.

    Dacă output_path există și resume=True, observațiile deja
    calculate sunt sărite, permițând reluarea calculului.
    """

    required = {
        "timestamp_utc",
        "segment_id",
        "air_temp_c",
        "wind_speed_ms",
        "wind_direction_deg",
        "solar_direct_normal_wm2",
        "solar_diffuse_wm2",
    }

    missing = required - set(weather_df.columns)

    if missing:
        raise ValueError(
            f"Lipsesc coloanele obligatorii: {sorted(missing)}"
        )

    if checkpoint_every < 1:
        raise ValueError("checkpoint_every trebuie să fie pozitiv.")

    data = weather_df.copy().reset_index(drop=True)

    if limit is not None:
        data = data.head(limit).copy()

    segments = load_segments()

    unknown_segments = set(data["segment_id"].unique()) - set(segments)

    if unknown_segments:
        raise ValueError(
            f"Segmente necunoscute: {unknown_segments}"
        )

    # Construim conductorul și span-urile o singură dată.
    conductor = create_conductor()

    spans = {
        segment_id: create_span(segment, conductor)
        for segment_id, segment in segments.items()
    }

    output_path = output_path or (
        f"{PROJECT_DIR}/data/processed/ampacity_training.csv"
    )

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    key_columns = ["timestamp_utc", "segment_id"]
    completed = set()

    # Recuperăm observațiile calculate anterior.
    if resume and os.path.exists(output_path):
        existing = pd.read_csv(
            output_path,
            usecols=key_columns + ["target_ampacity_a"],
        )

        completed = set(
            zip(
                existing["timestamp_utc"].astype(str),
                existing["segment_id"].astype(str),
            )
        )

        print(
            f"Checkpoint găsit: {len(completed):,} observații deja calculate."
        )

    pending = []

    try:
        from tqdm.auto import tqdm
        iterator = tqdm(
            data.iterrows(),
            total=len(data),
            desc="Calcul ampacitate",
        )
    except ImportError:
        iterator = data.iterrows()

    for index, row in iterator:
        key = (
            str(row["timestamp_utc"]),
            str(row["segment_id"]),
        )

        if key in completed:
            continue

        ampacity = calculate_ampacity(
            row,
            spans[row["segment_id"]],
        )

        result = row.to_dict()
        result["target_ampacity_a"] = ampacity

        pending.append(result)

        # Salvare periodică pentru protejarea progresului.
        if len(pending) >= checkpoint_every:
            batch = pd.DataFrame(pending)

            batch.to_csv(
                output_path,
                mode="a",
                header=not os.path.exists(output_path)
                or os.path.getsize(output_path) == 0,
                index=False,
            )

            completed.update(
                (
                    str(item["timestamp_utc"]),
                    str(item["segment_id"]),
                )
                for item in pending
            )

            pending.clear()

    # Salvăm și ultimul lot.
    if pending:
        batch = pd.DataFrame(pending)

        batch.to_csv(
            output_path,
            mode="a",
            header=not os.path.exists(output_path)
            or os.path.getsize(output_path) == 0,
            index=False,
        )

    print("\nCalcul finalizat.")
    print(f"Fișier: {output_path}")

    final_df = pd.read_csv(output_path)

    print(f"Total observații salvate: {len(final_df):,}")

    return final_df
'''

PHYSICS_FILE.write_text(code, encoding="utf-8")

print("physics_engine.py actualizat:", PHYSICS_FILE)
print("Dimensiune:", PHYSICS_FILE.stat().st_size, "bytes")


physics_engine.py actualizat: /content/Smarthack/gridpilot-ai/src/physics_engine.py
Dimensiune: 7732 bytes


In [67]:

import importlib
import src.physics_engine as pe

importlib.reload(pe)

weather_df = pd.read_csv(
    f"{pe.PROJECT_DIR}/data/processed/weather.csv"
)

test_df = pe.generate_training_dataset(
    weather_df,
    output_path=f"{pe.PROJECT_DIR}/data/processed/ampacity_test.csv",
    limit=30,
    checkpoint_every=10,
    resume=False,
)

print(test_df[[
    "segment_id",
    "timestamp_utc",
    "target_ampacity_a"
]].head(10))

print("\nStatistici:")
print(test_df["target_ampacity_a"].describe())


Calcul ampacitate:   0%|          | 0/30 [00:00<?, ?it/s]


Calcul finalizat.
Fișier: /content/Smarthack/gridpilot-ai/data/processed/ampacity_test.csv
Total observații salvate: 30
  segment_id         timestamp_utc  target_ampacity_a
0         S1  2022-01-01T06:00:00Z        1577.270508
1         S2  2022-01-01T06:00:00Z        1243.286133
2         S3  2022-01-01T06:00:00Z        1561.157227
3         S1  2022-01-01T07:00:00Z        1575.073242
4         S2  2022-01-01T07:00:00Z        1246.215820
5         S3  2022-01-01T07:00:00Z        1561.157227
6         S1  2022-01-01T08:00:00Z        1564.086914
7         S2  2022-01-01T08:00:00Z        1249.877930
8         S3  2022-01-01T08:00:00Z        1519.409180
9         S1  2022-01-01T09:00:00Z        1520.141602

Statistici:
count      30.000000
mean     1412.646484
std       142.480472
min      1134.887695
25%      1283.752441
50%      1423.828125
75%      1543.029785
max      1581.665039
Name: target_ampacity_a, dtype: float64


In [69]:

import importlib
import src.physics_engine as pe

importlib.reload(pe)

weather_df = pd.read_csv(
    f"{pe.PROJECT_DIR}/data/processed/weather.csv"
)

ampacity_df = pe.generate_training_dataset(
    weather_df,
    output_path=f"{pe.PROJECT_DIR}/data/processed/ampacity_training.csv",
    checkpoint_every=2000,
    resume=True,
)

print("\nDistribuția ampacității:")
print(ampacity_df["target_ampacity_a"].describe())

print("\nObservații pe segment:")
print(ampacity_df.groupby("segment_id")["target_ampacity_a"].agg(
    ["count", "mean", "std", "min", "max"]
))


Checkpoint găsit: 32,000 observații deja calculate.


Calcul ampacitate:   0%|          | 0/105165 [00:00<?, ?it/s]


Calcul finalizat.
Fișier: /content/Smarthack/gridpilot-ai/data/processed/ampacity_training.csv
Total observații salvate: 105,165

Distribuția ampacității:
count    105165.000000
mean       1349.025969
std         192.011097
min         559.936523
25%        1216.918945
50%        1363.403320
75%        1495.971680
max        1779.418945
Name: target_ampacity_a, dtype: float64

Observații pe segment:
            count         mean         std         min          max
segment_id                                                         
S1          35055  1349.515344  189.620910  576.049805  1761.840820
S2          35055  1370.543843  192.276765  559.936523  1779.418945
S3          35055  1327.018720  191.658133  601.684570  1758.178711


In [70]:
%cd /content/Smarthack/gridpilot-ai


/content/Smarthack/gridpilot-ai


In [71]:
!git status

On branch main
Your branch is behind 'origin/main' by 8 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py
	modified:   src/physics_engine.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	config/
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv
	data/processed/weather.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [72]:
!git stash push -u -m "ML Engineer ampacity work"

Saved working directory and index state On main: ML Engineer ampacity work


In [73]:
!git pull --ff-only origin main

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 4), reused 10 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 3.17 KiB | 1.06 MiB/s, done.
From https://github.com/Fandrei19/Smarthack
 * branch            main       -> FETCH_HEAD
   a866770..a547c75  main       -> origin/main
Updating 1897616..a547c75
Fast-forward
 .gitignore                                         |      1 +
 gridpilot-ai/README.md                             |     25 +
 gridpilot-ai/config/conductor_config.json          |     48 +
 gridpilot-ai/config/line_segments.json             |      8 +
 gridpilot-ai/data/demo/line_segments_mock.json     |      8 +
 gridpilot-ai/data/processed/predictions.csv        | 105193 ++++++++++++++++++
 gridpilot-ai/data/processed/weather.csv            | 105193 ++++++++++++++++++
 gridpilot-ai/data/raw/.gitkeep                     |      0
 gridpilot-ai/evaluati

In [74]:
!git switch -c ml-engineer-ampacity

Switched to a new branch 'ml-engineer-ampacity'


In [75]:
!git stash apply

Auto-merging gridpilot-ai/src/data_pipeline.py
Auto-merging gridpilot-ai/src/physics_engine.py
CONFLICT (content): Merge conflict in gridpilot-ai/src/physics_engine.py
gridpilot-ai/config/line_segments.json already exists, no checkout
gridpilot-ai/data/processed/weather.csv already exists, no checkout
error: could not restore untracked files from stash
On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Unmerged paths:
  (use "git restore --staged <file>..." to unstage)
  (use "git add <file>..." to mark resolution)
	both modified:   src/physics_engine.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv



In [76]:
!git status

On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Unmerged paths:
  (use "git restore --staged <file>..." to unstage)
  (use "git add <file>..." to mark resolution)
	both modified:   src/physics_engine.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv



In [77]:
%cd /content/Smarthack/gridpilot-ai

/content/Smarthack/gridpilot-ai


In [78]:

!git status
!git diff -- src/physics_engine.py

On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Unmerged paths:
  (use "git restore --staged <file>..." to unstage)
  (use "git add <file>..." to mark resolution)
	both modified:   src/physics_engine.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv

diff --cc gridpilot-ai/src/physics_engine.py
index 1992e10,21072c1..0000000
--- a/gridpilot-ai/src/physics_engine.py
+++ b/gridpilot-ai/src/physics_engine.py
@@@ -1,341 -1,311 +1,655 @@@
++<<<<<<< Updated upstream
 +"""Physics engine: CIGRE TB 601 steady-state ampacity via Statnett LineRate.
 +
 +SIMULATION ONLY. All conductor / span / static-rating parameters come from
 +``config/conductor_config.json`` and are demonstrative. No real PPC grid data.
 +
 +Units 

In [79]:

from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/Smarthack/gridpilot-ai")

# Backup pentru fișierul aflat în conflict
conflict_file = PROJECT / "src/physics_engine.py"
backup_file = PROJECT / "physics_engine_conflict_backup.py"

backup_file.write_text(
    conflict_file.read_text(encoding="utf-8"),
    encoding="utf-8"
)

print("Backup salvat:", backup_file)

# Verificăm configurațiile
for filename in [
    "config/conductor_config.json",
    "config/line_segments.json",
]:
    path = PROJECT / filename
    print(f"\n--- {filename} ---")

    if path.exists():
        print(json.dumps(
            json.loads(path.read_text(encoding="utf-8")),
            indent=2,
            ensure_ascii=False
        ))
    else:
        print("Fișier inexistent")

# Verificăm dataseturile
for filename in [
    "data/processed/weather.csv",
    "data/processed/ampacity_training.csv",
]:
    path = PROJECT / filename
    print(f"\n--- {filename} ---")

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Coloane:", df.columns.tolist())
        if "target_ampacity_a" in df.columns:
            print(df["target_ampacity_a"].describe())
    else:
        print("Fișier inexistent")

# Verificăm dacă alte module folosesc funcțiile Physics Engine
import subprocess

result = subprocess.run(
    ["grep", "-R", "-n", "-E",
     "generate_labels|generate_training_dataset|calculate_ampacity|compute_static_rating|load_conductor_config",
     "src"],
    cwd=PROJECT,
    capture_output=True,
    text=True
)

print("\n--- Utilizări în proiect ---")
print(result.stdout or "Nu au fost găsite utilizări.")


Backup salvat: /content/Smarthack/gridpilot-ai/physics_engine_conflict_backup.py

--- config/conductor_config.json ---
{
  "_meta": {
    "owner": "Participant 2 (physics_engine)",
    "status": "DEMONSTRATIVE / SIMULATED. Not a PPC / Transelectrica asset. No real grid measurements.",
    "conductor_source": "Conductor, span and weather parameters of CIGRE TB 601 'Example B' exactly as reproduced in the official LineRate example https://statnett.github.io/linerate/auto_examples/plot_cigre.html (the docs describe the example as taken from CIGRE TB 601, Example B). The docs do NOT name the conductor type, so we do not name it either. Roughly a 28.1 mm ACSR-class conductor.",
    "units_source": "Units from https://statnett.github.io/linerate/api/types.html (diameters m, temperatures degC, resistance ohm/m, albedo/emissivity dimensionless).",
    "not_in_linerate_docs": "Everything under 'operating', 'span' and 'static_rating' is OUR simulation assumption (not from LineRate/CIGRE). Team m

In [80]:
import pandas as pd
from pathlib import Path

base = Path("/content/Smarthack/gridpilot-ai")
weather_path = base / "data/processed/weather.csv"
training_path = base / "data/processed/ampacity_training.csv"

weather = pd.read_csv(weather_path)
training = pd.read_csv(training_path)

print("=== SEGMENTE ===")
print("weather.csv:")
print(weather["segment_id"].value_counts(dropna=False).sort_index())

print("\nampacity_training.csv:")
print(training["segment_id"].value_counts(dropna=False).sort_index())

print("\n=== INTERVALE TEMPORALE ===")
for name, df in [("weather", weather), ("training", training)]:
    print(
        name,
        "min:", df["timestamp_utc"].min(),
        "max:", df["timestamp_utc"].max()
    )

print("\n=== OBSERVAȚII ===")
print("Rânduri weather:", len(weather))
print("Rânduri training:", len(training))
print("Chei duplicate weather:",
      weather.duplicated(["timestamp_utc", "segment_id"]).sum())
print("Chei duplicate training:",
      training.duplicated(["timestamp_utc", "segment_id"]).sum())

weather_keys = set(zip(weather["timestamp_utc"], weather["segment_id"]))
training_keys = set(zip(training["timestamp_utc"], training["segment_id"]))

print("Observații fără etichetă:", len(weather_keys - training_keys))
print("Etichete fără observație meteo:", len(training_keys - weather_keys))

print("\n=== COLOANE SOLARE ===")
print("weather:", [c for c in weather.columns if "solar" in c])
print("training:", [c for c in training.columns if "solar" in c])

print("\n=== ML MODEL ===")
ml_path = base / "src/ml_model.py"
lines = ml_path.read_text().splitlines()

for start, end in [(130, 200), (280, 330)]:
    print(f"\n--- ml_model.py: {start}-{end} ---")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start-1, min(end, len(lines)))
    ))

=== SEGMENTE ===
weather.csv:
segment_id
SEG_1    35064
SEG_2    35064
SEG_3    35064
Name: count, dtype: int64

ampacity_training.csv:
segment_id
S1    35055
S2    35055
S3    35055
Name: count, dtype: int64

=== INTERVALE TEMPORALE ===
weather min: 2022-01-01T00:00:00Z max: 2025-12-31T23:00:00Z
training min: 2022-01-01T06:00:00Z max: 2025-12-31T22:00:00Z

=== OBSERVAȚII ===
Rânduri weather: 105192
Rânduri training: 105165
Chei duplicate weather: 0
Chei duplicate training: 0
Observații fără etichetă: 105192
Etichete fără observație meteo: 105165

=== COLOANE SOLARE ===
weather: ['solar_radiation']
training: ['solar_radiation', 'solar_direct_normal_wm2', 'solar_diffuse_wm2']

=== ML MODEL ===

--- ml_model.py: 130-200 ---
130:     for col, key in (("latitude", "latitude"), ("longitude", "longitude"), ("line_azimuth_deg", "azimuth_deg")):
131:         out[col] = seg_ids.map(lambda s: float(segments[s][key]))
132:     delta = np.abs((out["wind_direction_deg"] - out["line_azimuth_deg"] + 

In [81]:
print("=== VERSIUNEA CURRENT BRANCH ===")
!git show :2:src/physics_engine.py | head -35

print("\n=== VERSIUNEA STASH ===")
!git show :3:src/physics_engine.py | head -35

=== VERSIUNEA CURRENT BRANCH ===
fatal: path 'gridpilot-ai/src/physics_engine.py' is in the index, but not 'src/physics_engine.py'
hint: Did you mean ':1:gridpilot-ai/src/physics_engine.py' aka ':1:./src/physics_engine.py'?

=== VERSIUNEA STASH ===
fatal: path 'gridpilot-ai/src/physics_engine.py' is in the index, but not 'src/physics_engine.py'
hint: Did you mean ':1:gridpilot-ai/src/physics_engine.py' aka ':1:./src/physics_engine.py'?


In [82]:
%cd /content/Smarthack/gridpilot-ai

print("=== VERSIUNEA CURRENT BRANCH ===")
!git show :2:src/physics_engine.py | head -35

print("\n=== VERSIUNEA STASH ===")
!git show :3:src/physics_engine.py | head -35

/content/Smarthack/gridpilot-ai
=== VERSIUNEA CURRENT BRANCH ===
fatal: path 'gridpilot-ai/src/physics_engine.py' is in the index, but not 'src/physics_engine.py'
hint: Did you mean ':1:gridpilot-ai/src/physics_engine.py' aka ':1:./src/physics_engine.py'?

=== VERSIUNEA STASH ===
fatal: path 'gridpilot-ai/src/physics_engine.py' is in the index, but not 'src/physics_engine.py'
hint: Did you mean ':1:gridpilot-ai/src/physics_engine.py' aka ':1:./src/physics_engine.py'?


In [83]:
%cd /content/Smarthack/gridpilot-ai

print("=== STAREA CONFLICTULUI ===")
!git status --short

print("\n=== VERSIUNEA CURRENT BRANCH (OURS) ===")
!git show :2:gridpilot-ai/src/physics_engine.py | head -40

print("\n=== VERSIUNEA STASH (THEIRS) ===")
!git show :3:gridpilot-ai/src/physics_engine.py | head -40

/content/Smarthack/gridpilot-ai
=== STAREA CONFLICTULUI ===
M  data/processed/dataset.csv
M  src/data_pipeline.py
UU src/physics_engine.py
?? data/processed/ampacity_test.csv
?? data/processed/ampacity_training.csv
?? data/processed/dataset_before_solar.csv
?? physics_engine_conflict_backup.py

=== VERSIUNEA CURRENT BRANCH (OURS) ===
"""Physics engine: CIGRE TB 601 steady-state ampacity via Statnett LineRate.

SIMULATION ONLY. All conductor / span / static-rating parameters come from
``config/conductor_config.json`` and are demonstrative. No real PPC grid data.

Units (everything in this module):
    temperatures  degC          wind speed      m/s
    wind direction  degrees clockwise from north (meteorological, as in weather.csv)
    solar_radiation W/m^2 (global horizontal irradiance)
    ampacity      A             lengths / altitude  m
    timestamps    UTC (tz-naive input is *assumed* UTC)

LineRate interface used (verified against the official docs, NOT executed in the
authoring 

In [84]:
%cd /content/Smarthack/gridpilot-ai

# Păstrează versiunea echipei pentru physics_engine.py
!git checkout --ours src/physics_engine.py

# Marchează conflictul ca rezolvat
!git add src/physics_engine.py

# Verifică starea
!git status --short

/content/Smarthack/gridpilot-ai
Updated 1 path from the index
M  data/processed/dataset.csv
M  src/data_pipeline.py
?? data/processed/ampacity_test.csv
?? data/processed/ampacity_training.csv
?? data/processed/dataset_before_solar.csv
?? physics_engine_conflict_backup.py


In [85]:
%cd /content/Smarthack/gridpilot-ai

import src.physics_engine as pe

print("Funcții disponibile:")
for name in [
    "load_conductor_config",
    "load_segments",
    "calculate_ampacity",
    "compute_static_rating",
    "generate_labels",
    "solar_sin_altitude",
]:
    print(name, ":", hasattr(pe, name))

print("\nConfigurație:")
config = pe.load_conductor_config()
segments = pe.load_segments()

print("Segmente:", [s["segment_id"] for s in segments])
print("Conductor:", config["conductor"]["conductor_diameter"])

/content/Smarthack/gridpilot-ai
Funcții disponibile:
load_conductor_config : False
load_segments : True
calculate_ampacity : True
compute_static_rating : False
generate_labels : False
solar_sin_altitude : False

Configurație:


AttributeError: module 'src.physics_engine' has no attribute 'load_conductor_config'

In [86]:
%cd /content/Smarthack/gridpilot-ai

import importlib
import inspect
import src.physics_engine as pe

importlib.reload(pe)

print("FIȘIER ÎNCĂRCAT:")
print(pe.__file__)

print("\nFUNCȚII DEFINITE:")
print([
    name for name, obj in inspect.getmembers(pe, inspect.isfunction)
    if obj.__module__ == pe.__name__
])

print("\nPRIMELE 100 DE LINII:")
with open(pe.__file__, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100:
            break
        print(f"{i+1}: {line.rstrip()}")

/content/Smarthack/gridpilot-ai
FIȘIER ÎNCĂRCAT:
/content/Smarthack/gridpilot-ai/src/physics_engine.py

FUNCȚII DEFINITE:
['_build_towers', '_check_inputs', '_conductor_kwargs', '_import_linerate', 'calculate_ampacity', 'compute_static_rating', 'create_conductor', 'create_span', 'destination_point', 'estimate_clearness_ratio', 'generate_labels', 'generate_training_dataset', 'load_conductor_config', 'load_segments', 'solar_sin_altitude']

PRIMELE 100 DE LINII:
1: """Physics engine: CIGRE TB 601 steady-state ampacity via Statnett LineRate.
2: 
3: SIMULATION ONLY. All conductor / span / static-rating parameters come from
4: ``config/conductor_config.json`` and are demonstrative. No real PPC grid data.
5: 
6: Units (everything in this module):
7:     temperatures  degC          wind speed      m/s
8:     wind direction  degrees clockwise from north (meteorological, as in weather.csv)
9:     solar_radiation W/m^2 (global horizontal irradiance)
10:     ampacity      A             lengths / a

In [87]:
!git status --short
!git diff -- src/physics_engine.py
!grep -nE "^def |^class " src/physics_engine.py

M  data/processed/dataset.csv
M  src/data_pipeline.py
?? data/processed/ampacity_test.csv
?? data/processed/ampacity_training.csv
?? data/processed/dataset_before_solar.csv
?? physics_engine_conflict_backup.py
52:class PhysicsError(RuntimeError):
56:class LineRateNotAvailableError(ImportError):
65:def load_conductor_config(path: str | Path | None = None) -> dict[str, Any]:
79:def load_segments(path: str | Path) -> list[dict[str, Any]]:
99:def solar_sin_altitude(timestamps: pd.Series | pd.DatetimeIndex, latitude_deg: float,
122:def _import_linerate():
140:def _conductor_kwargs(conductor_cfg: Mapping[str, Any]) -> dict[str, Any]:
147:def _build_towers(lat: float, lon: float, azimuth_deg: float, length_m: float,
158:def estimate_clearness_ratio(ghi_wm2: float, sin_alt: float, altitude_m: float,
181:def _check_inputs(w: Mapping[str, Any]) -> None:
202:def calculate_ampacity(weather_features: Mapping[str, Any], conductor_config: Mapping[str, Any]) -> float:
282:def compute_static_rating(seg

In [88]:
%cd /content/Smarthack/gridpilot-ai

import importlib
import src.physics_engine as pe

importlib.reload(pe)

config = pe.load_conductor_config()
segments = pe.load_segments("config/line_segments.json")

print("Config încărcat:", config is not None)
print("Segmente:", [s["segment_id"] for s in segments])
print("Diametru conductor:", config["conductor"]["conductor_diameter"])
print("Lungime deschidere:", config["span"]["length_m"])
print("Temperatură maximă:", config["operating"]["max_conductor_temperature_c"])

/content/Smarthack/gridpilot-ai
Config încărcat: True
Segmente: ['SEG_1', 'SEG_2', 'SEG_3']
Diametru conductor: 0.0281
Lungime deschidere: 400.0
Temperatură maximă: 75.0


In [89]:
import pandas as pd

weather = pd.read_csv("data/processed/weather.csv")

print("Coloane:", weather.columns.tolist())
print("\nPrimele observații:")
print(weather.head(3).to_string(index=False))

print("\nFuncția de calcul:")
import inspect
print(inspect.signature(pe.calculate_ampacity))
print(inspect.signature(pe.generate_labels))


Coloane: ['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_direction_deg', 'solar_radiation', 'source', 'data_quality']

Primele observații:
       timestamp_utc segment_id  air_temp_c  wind_speed_ms  wind_direction_deg  solar_radiation source data_quality
2022-01-01T00:00:00Z      SEG_1         4.5           3.22                 276                0   ERA5           ok
2022-01-01T00:00:00Z      SEG_2         4.2           4.12                 284                0   ERA5           ok
2022-01-01T00:00:00Z      SEG_3         4.8           4.14                 278                0   ERA5           ok

Funcția de calcul:
(weather_features: 'Mapping[str, Any]', conductor_config: 'Mapping[str, Any]') -> 'float'
(weather: 'pd.DataFrame', segments: 'Sequence[Mapping[str, Any]]', conductor_config: 'Mapping[str, Any]', ampacity_fn: 'AmpacityFn' = <function calculate_ampacity at 0x7eb5407adbc0>, max_failure_fraction: 'float' = 0.2, log_every: 'int' = 20000) -> 'pd.DataFrame'


In [90]:
segment_map = {s["segment_id"]: s for s in segments}

row = weather.iloc[0]
segment = segment_map[row["segment_id"]]

features = row.to_dict()
features.update({
    "latitude": segment["latitude"],
    "longitude": segment["longitude"],
    "azimuth_deg": segment["azimuth_deg"],
})

ampacity = pe.calculate_ampacity(features, config)

print("Segment:", row["segment_id"])
print("Timestamp:", row["timestamp_utc"])
print("Ampacitate:", round(ampacity, 2), "A")

Segment: SEG_1
Timestamp: 2022-01-01T00:00:00Z
Ampacitate: 1417.24 A


In [91]:
%cd /content/Smarthack/gridpilot-ai
!git status
!git log -1 --oneline

/content/Smarthack/gridpilot-ai
On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv
	physics_engine_conflict_backup.py

a547c75 (HEAD -> ml-engineer-ampacity, origin/main, origin/HEAD, main) implement risk-aware decision engine


In [92]:
%cd /content/Smarthack/gridpilot-ai

import importlib
import pandas as pd
import src.physics_engine as pe
import src.ml_model as ml

importlib.reload(pe)
importlib.reload(ml)

weather = pd.read_csv("data/processed/weather.csv")
config = pe.load_conductor_config()
segments = pe.load_segments("config/line_segments.json")

test_ds = ml.build_training_dataset(
    weather=weather.head(12).copy(),
    segments=segments,
    conductor_config=config
)

print("Shape:", test_ds.shape)
print("Segmente:", test_ds["segment_id"].value_counts().to_dict())
print("Coloane:", test_ds.columns.tolist())
print("\nPrimele rezultate:")
print(test_ds.head(12).to_string(index=False))

print("\nStatistici ampacitate:")
print(test_ds["target_ampacity_a"].describe())

/content/Smarthack/gridpilot-ai
Shape: (12, 16)
Segmente: {'SEG_1': 4, 'SEG_2': 4, 'SEG_3': 4}
Coloane: ['timestamp_utc', 'segment_id', 'air_temp_c', 'wind_speed_ms', 'wind_angle_of_attack_deg', 'crosswind_ms', 'parallel_wind_ms', 'solar_radiation', 'sin_solar_altitude', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'line_azimuth_deg', 'data_quality', 'target_ampacity_a']

Primele rezultate:
            timestamp_utc segment_id  air_temp_c  wind_speed_ms  wind_angle_of_attack_deg  crosswind_ms  parallel_wind_ms  solar_radiation  sin_solar_altitude  hour_sin  hour_cos  doy_sin  doy_cos  line_azimuth_deg data_quality  target_ampacity_a
2022-01-01 00:00:00+00:00      SEG_1         4.5           3.22                      24.0      1.309692          2.941616                0           -0.857664  0.000000  1.000000 0.017202 0.999852              72.0           ok        1417.236328
2022-01-01 00:00:00+00:00      SEG_2         4.2           4.12                      33.0      2.243913        

In [93]:
full_ds = ml.build_training_dataset(
    weather=weather,
    segments=segments,
    conductor_config=config,
    out_path="data/processed/ampacity_training.csv",
    failures_path="data/processed/ampacity_failures.csv"
)

print("Shape final:", full_ds.shape)
print("\nDistribuție segmente:")
print(full_ds["segment_id"].value_counts())

print("\nStatistici target:")
print(full_ds["target_ampacity_a"].describe())

print("\nValori lipsă:")
print(full_ds.isna().sum())

print("\nEtichete nevalide:")
print(
    (~full_ds["target_ampacity_a"].apply(
        lambda x: pd.notna(x) and x > 0 and x < float("inf")
    )).sum()
)

Shape final: (105192, 16)

Distribuție segmente:
segment_id
SEG_1    35064
SEG_2    35064
SEG_3    35064
Name: count, dtype: int64

Statistici target:
count    105192.000000
mean       1367.434110
std         194.191071
min         555.908203
25%        1237.060547
50%        1386.474609
75%        1515.380859
max        1790.283203
Name: target_ampacity_a, dtype: float64

Valori lipsă:
timestamp_utc               0
segment_id                  0
air_temp_c                  0
wind_speed_ms               0
wind_angle_of_attack_deg    0
crosswind_ms                0
parallel_wind_ms            0
solar_radiation             0
sin_solar_altitude          0
hour_sin                    0
hour_cos                    0
doy_sin                     0
doy_cos                     0
line_azimuth_deg            0
data_quality                0
target_ampacity_a           0
dtype: int64

Etichete nevalide:
0


In [94]:
%cd /content/Smarthack/gridpilot-ai

!git status

/content/Smarthack/gridpilot-ai
On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv
	physics_engine_conflict_backup.py



In [95]:
!git status
!git log -1 --oneline
!git ls-files src/ml_model.py
!git log -1 --oneline -- src/ml_model.py

On branch ml-engineer-ampacity
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   data/processed/dataset.csv
	modified:   src/data_pipeline.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/ampacity_test.csv
	data/processed/ampacity_training.csv
	data/processed/dataset_before_solar.csv
	physics_engine_conflict_backup.py

a547c75 (HEAD -> ml-engineer-ampacity, origin/main, origin/HEAD, main) implement risk-aware decision engine
src/ml_model.py
7c614e6 Add physics engine, ML model, evaluation and tests (Participant 2)


In [96]:
!git diff HEAD -- src/ml_model.py
!git diff HEAD -- src/physics_engine.py

In [97]:
!git push origin ml-engineer-ampacity

fatal: could not read Username for 'https://github.com': No such device or address


In [98]:
import subprocess
from getpass import getpass

username = input("GitHub username: ")
token = getpass("GitHub token: ")

# Păstrează credentialele în memoria runtime-ului, timp de o oră
subprocess.run(
    ["git", "config", "--global", "credential.helper",
     "cache --timeout=3600"],
    check=True
)

# Transmite credentialele fără să le afișeze
credential = (
    "protocol=https\n"
    "host=github.com\n"
    f"username={username}\n"
    f"password={token}\n\n"
)

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

del token

print("Autentificare configurată pentru acest runtime.")

GitHub username: Fandrei19
GitHub token: ··········
Autentificare configurată pentru acest runtime.


In [99]:
!git push origin ml-engineer-ampacity

Enumerating objects: 27, done.
Counting objects: 100% (26/26), done.
Delta compression using up to 2 threads
Compressing objects: 100% (18/18), done.
Writing objects: 100% (18/18), 6.05 KiB | 516.00 KiB/s, done.
Total 18 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 2 local objects.
remote: 
remote: Create a pull request for 'ml-engineer-ampacity' on GitHub by visiting:
remote:      https://github.com/Fandrei19/Smarthack/pull/new/ml-engineer-ampacity
remote: 
To https://github.com/Fandrei19/Smarthack
 * [new branch]      ml-engineer-ampacity -> ml-engineer-ampacity


In [100]:
%cd /content/Smarthack/gridpilot-ai

import inspect
import src.ml_model as ml

print("=== SEMNĂTURA TRAIN_PIPELINE ===")
print(inspect.signature(ml.train_pipeline))

print("\n=== IMPLEMENTARE TRAIN_PIPELINE ===")
print(inspect.getsource(ml.train_pipeline))

/content/Smarthack/gridpilot-ai
=== SEMNĂTURA TRAIN_PIPELINE ===
(weather: 'pd.DataFrame', segments: 'Sequence[Mapping[str, Any]]', conductor_config: 'Mapping[str, Any]', cfg: 'TrainConfig' = TrainConfig(random_state=42, train_frac=0.7, val_frac=0.15, gap_hours=24.0, low_alpha=0.05, high_alpha=0.95, max_iter=300, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0, backtest_folds=4), ampacity_fn: 'pe.AmpacityFn' = <function calculate_ampacity at 0x7eb5407adb20>, dataset_path: 'str | Path | None' = None, failures_path: 'str | Path | None' = None) -> 'tuple[dict[str, Any], pd.DataFrame, dict[str, Any]]'

=== IMPLEMENTARE TRAIN_PIPELINE ===
def train_pipeline(weather: pd.DataFrame, segments: Sequence[Mapping[str, Any]],
                   conductor_config: Mapping[str, Any], cfg: TrainConfig = TrainConfig(),
                   ampacity_fn: pe.AmpacityFn = pe.calculate_ampacity,
                   dataset_path: str | Path | None = None, failures_path: str | Pa

In [101]:
import inspect
import src.ml_model as ml

print("=== FIT MODELS ===")
print(inspect.signature(ml.fit_models))
print(inspect.getsource(ml.fit_models))

print("\n=== CHRONOLOGICAL SPLIT ===")
print(inspect.signature(ml.ev.chronological_split))

print("\n=== ROLLING ORIGIN BACKTEST ===")
print(inspect.signature(ml.rolling_origin_backtest))

print("\n=== STATIC RATING ===")
print(inspect.signature(ml.pe.compute_static_rating))

=== FIT MODELS ===
(train: 'pd.DataFrame', val: 'pd.DataFrame', cfg: 'TrainConfig') -> 'dict[str, Any]'
def fit_models(train: pd.DataFrame, val: pd.DataFrame, cfg: TrainConfig) -> dict[str, Any]:
    """Fit baselines + HGB models on ``train``; calibrate lower bounds on ``val`` (later in time)."""
    X, y = train[FEATURE_COLUMNS], train["target_ampacity_a"].to_numpy()
    Xv, yv = val[FEATURE_COLUMNS], val["target_ampacity_a"].to_numpy()
    m: dict[str, Any] = {
        "dummy_median": DummyRegressor(strategy="median").fit(X, y),
        "dummy_q05": DummyRegressor(strategy="quantile", quantile=cfg.low_alpha).fit(X, y),
        "dummy_q95": DummyRegressor(strategy="quantile", quantile=cfg.high_alpha).fit(X, y),
        "linear": LinearRegression().fit(X, y),
        "hgb_point": _hgb(cfg, loss="squared_error").fit(X, y),
        "hgb_q05": _hgb(cfg, loss="quantile", quantile=cfg.low_alpha).fit(X, y),
        "hgb_q50": _hgb(cfg, loss="quantile", quantile=0.5).fit(X, y),
        "hgb_q

In [102]:
print("=== EVALUARE MODELE ===")
print(inspect.getsource(ml._evaluate_split))

print("\n=== BACKTESTING ===")
print(inspect.getsource(ml.rolling_origin_backtest))

=== EVALUARE MODELE ===
def _evaluate_split(df: pd.DataFrame, models: Mapping[str, Any], static: Mapping[str, float]) -> dict[str, Any]:
    X, y = df[FEATURE_COLUMNS], df["target_ampacity_a"].to_numpy()
    dq = predict_quantiles(models, X)
    lin = models["linear"].predict(X)
    per_model = {
        "baseline_dummy_median(+const q05/q95)": ev.evaluate_predictions(y, {
            "q50": models["dummy_median"].predict(X), "q05": models["dummy_q05"].predict(X),
            "q95": models["dummy_q95"].predict(X)}),
        "baseline_linear(+conformal q05)": ev.evaluate_predictions(y, {
            "q50": lin, "q05": np.maximum(lin - models["linear_shift_a"], 0.0)}),
        "hgb_point(squared_error)": ev.evaluate_predictions(y, {"q50": models["hgb_point"].predict(X)}),
        "hgb_quantile_raw_q05": ev.evaluate_predictions(y, {
            "q50": models["hgb_q50"].predict(X), "q05": models["hgb_q05"].predict(X),
            "q95": models["hgb_q95"].predict(X)}),
        "hgb_quantile

In [103]:
from pathlib import Path
import shutil

path = Path("src/ml_model.py")
source = path.read_text()

# Backup
shutil.copy2(path, "/tmp/ml_model_before_prebuilt.py")

old_signature = """                   dataset_path: str | Path | None = None, failures_path: str | Path | None = None
                   ) -> tuple[dict[str, Any], pd.DataFrame, dict[str, Any]]:"""

new_signature = """                   dataset_path: str | Path | None = None, failures_path: str | Path | None = None,
                   prebuilt_dataset: pd.DataFrame | None = None
                   ) -> tuple[dict[str, Any], pd.DataFrame, dict[str, Any]]:"""

old_dataset = """    ds = build_training_dataset(weather, segments, conductor_config, ampacity_fn, dataset_path, failures_path)"""

new_dataset = """    if prebuilt_dataset is not None:
        ds = prebuilt_dataset.copy()
        required = set(FEATURE_COLUMNS + [
            "timestamp_utc", "segment_id", "target_ampacity_a"
        ])
        missing = required - set(ds.columns)
        if missing:
            raise ValueError(f"Dataset preconstruit incomplet. Lipsesc: {sorted(missing)}")
        if ds.empty:
            raise ValueError("Datasetul preconstruit este gol.")
    else:
        ds = build_training_dataset(
            weather, segments, conductor_config,
            ampacity_fn, dataset_path, failures_path
        )"""

assert source.count(old_signature) == 1, "Nu am găsit semnătura exactă."
assert source.count(old_dataset) == 1, "Nu am găsit instrucțiunea de construire a datasetului."

source = source.replace(old_signature, new_signature)
source = source.replace(old_dataset, new_dataset)

path.write_text(source)

print("train_pipeline modificat cu succes!")

train_pipeline modificat cu succes!


In [104]:
import importlib
import inspect
import src.ml_model as ml

importlib.reload(ml)

print(inspect.signature(ml.train_pipeline))

(weather: 'pd.DataFrame', segments: 'Sequence[Mapping[str, Any]]', conductor_config: 'Mapping[str, Any]', cfg: 'TrainConfig' = TrainConfig(random_state=42, train_frac=0.7, val_frac=0.15, gap_hours=24.0, low_alpha=0.05, high_alpha=0.95, max_iter=300, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0, backtest_folds=4), ampacity_fn: 'pe.AmpacityFn' = <function calculate_ampacity at 0x7eb5407adb20>, dataset_path: 'str | Path | None' = None, failures_path: 'str | Path | None' = None, prebuilt_dataset: 'pd.DataFrame | None' = None) -> 'tuple[dict[str, Any], pd.DataFrame, dict[str, Any]]'


In [105]:
bundle, trained_ds, report = ml.train_pipeline(
    weather=weather,
    segments=segments,
    conductor_config=config,
    prebuilt_dataset=full_ds
)

print("\n=== ANTRENARE FINALIZATĂ ===")
print("Dataset:", trained_ds.shape)
print("Modele salvate în bundle:", list(bundle["models"].keys()))

print("\n=== CONFIGURAȚIE SPLIT ===")
print(report["split"])

print("\n=== REZULTATE TEST ===")
for name, metrics in report["test"]["models"].items():
    print(f"\n{name}")
    print(metrics)

print("\n=== BACKTESTING ===")
for fold in report["rolling_origin_backtest"]:
    print(fold)


=== ANTRENARE FINALIZATĂ ===
Dataset: (105192, 16)
Modele salvate în bundle: ['hgb_q05', 'hgb_q50', 'hgb_q95', 'q05_shift_a']

=== CONFIGURAȚIE SPLIT ===
{'train': {'rows': 73632, 'start': '2022-01-01 00:00:00+00:00', 'end': '2024-10-19 15:00:00+00:00'}, 'val': {'rows': 15708, 'start': '2024-10-20 16:00:00+00:00', 'end': '2025-05-26 19:00:00+00:00'}, 'test': {'rows': 15708, 'start': '2025-05-27 20:00:00+00:00', 'end': '2025-12-31 23:00:00+00:00'}, 'gap_hours': 24.0}

=== REZULTATE TEST ===

baseline_dummy_median(+const q05/q95)
{'n': 15708, 'mae_a': 150.3152752578304, 'rmse_a': 191.7765199443451, 'pinball_q50': 75.1576376289152, 'pinball_q05': 21.995926389021516, 'lower_bound_coverage': 0.9386299974535268, 'overestimation_frequency': 0.06137000254647314, 'overestimation_count': 964, 'nominal_overestimation_frequency': 0.05, 'mean_overestimation_a': 101.70483094528008, 'max_overestimation_a': 446.77734375, 'mean_conservatism_a': 342.3368523318808, 'pinball_q95': 16.463626466211498, 'up

In [106]:
import inspect

print("=== STRUCTURA BUNDLE ===")
print(bundle.keys())

print("\n=== CONFIGURAȚIA FEATURE-URILOR ===")
print(bundle.get("features"))

print("\n=== FUNCȚIA DE EVALUARE ===")
print(inspect.getsource(ml._evaluate_split))

=== STRUCTURA BUNDLE ===
dict_keys(['format_version', 'feature_columns', 'models', 'segments', 'static_rating_a', 'training_ranges', 'metadata'])

=== CONFIGURAȚIA FEATURE-URILOR ===
None

=== FUNCȚIA DE EVALUARE ===
def _evaluate_split(df: pd.DataFrame, models: Mapping[str, Any], static: Mapping[str, float]) -> dict[str, Any]:
    X, y = df[FEATURE_COLUMNS], df["target_ampacity_a"].to_numpy()
    dq = predict_quantiles(models, X)
    lin = models["linear"].predict(X)
    per_model = {
        "baseline_dummy_median(+const q05/q95)": ev.evaluate_predictions(y, {
            "q50": models["dummy_median"].predict(X), "q05": models["dummy_q05"].predict(X),
            "q95": models["dummy_q95"].predict(X)}),
        "baseline_linear(+conformal q05)": ev.evaluate_predictions(y, {
            "q50": lin, "q05": np.maximum(lin - models["linear_shift_a"], 0.0)}),
        "hgb_point(squared_error)": ev.evaluate_predictions(y, {"q50": models["hgb_point"].predict(X)}),
        "hgb_quantile_raw_

In [107]:
import inspect

print("=== FEATURE COLUMNS ===")
print(bundle["feature_columns"])

print("\n=== PREDICT QUANTILES ===")
print(inspect.getsource(ml.predict_quantiles))

print("\n=== MODELE DIN BUNDLE ===")
for name, model in bundle["models"].items():
    print(name, type(model).__name__)

=== FEATURE COLUMNS ===
['air_temp_c', 'wind_speed_ms', 'wind_angle_of_attack_deg', 'crosswind_ms', 'parallel_wind_ms', 'solar_radiation', 'sin_solar_altitude', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'line_azimuth_deg']

=== PREDICT QUANTILES ===
def predict_quantiles(models: Mapping[str, Any], X: pd.DataFrame) -> dict[str, np.ndarray]:
    """Deployed quantile estimates: calibrated lower bound, median, upper; ordered q05<=q50<=q95.

    Quantile crossing (independent models) is repaired by sorting the three values per row,
    which can only lower q05 / raise q95 (conservative). ``crossing`` flags repaired rows.
    """
    q05 = models["hgb_q05"].predict(X) - models["q05_shift_a"]
    q50, q95 = models["hgb_q50"].predict(X), models["hgb_q95"].predict(X)
    stacked = np.vstack([q05, q50, q95])
    crossing = np.any(np.diff(stacked, axis=0) < 0, axis=0)
    s = np.sort(stacked, axis=0)
    return {"q05": np.maximum(s[0], 0.0), "q50": np.maximum(s[1], 0.0), "q95": np.maximum(s[2

In [108]:
import numpy as np
import pandas as pd

# Reconstruim exact setul de test pe baza intervalului din raport
test_start = pd.to_datetime(report["split"]["test"]["start"], utc=True)
test_end = pd.to_datetime(report["split"]["test"]["end"], utc=True)

timestamps = pd.to_datetime(trained_ds["timestamp_utc"], utc=True)

test_df = trained_ds[
    (timestamps >= test_start) &
    (timestamps <= test_end)
].copy()

X_test = test_df[ml.FEATURE_COLUMNS]

models = bundle["models"]

# Predicții brute, înainte de sortare
q05_raw = models["hgb_q05"].predict(X_test) - models["q05_shift_a"]
q50_raw = models["hgb_q50"].predict(X_test)
q95_raw = models["hgb_q95"].predict(X_test)

stacked = np.vstack([q05_raw, q50_raw, q95_raw])

# Tipurile de inversări
q05_gt_q50 = q05_raw > q50_raw
q50_gt_q95 = q50_raw > q95_raw
q05_gt_q95 = q05_raw > q95_raw

print("=== ANALIZA CROSSING ===")
print("Total observații:", len(test_df))

print(f"Q05 > Q50: {q05_gt_q50.mean()*100:.2f}%")
print(f"Q50 > Q95: {q50_gt_q95.mean()*100:.2f}%")
print(f"Q05 > Q95: {q05_gt_q95.mean()*100:.2f}%")

# Efectul sortării asupra limitei inferioare
q05_sorted = np.maximum(np.min(stacked, axis=0), 0.0)
q05_before = np.maximum(q05_raw, 0.0)

reduction = q05_before - q05_sorted

print("\n=== EFECTUL SORTĂRII ASUPRA Q05 ===")
print(f"Observații modificate: {(reduction > 1e-9).mean()*100:.2f}%")
print(f"Reducere medie Q05: {reduction.mean():.3f} A")
print(f"Reducere maximă Q05: {reduction.max():.3f} A")

print("\n=== VERIFICARE ORDINE DUPĂ SORTARE ===")
sorted_values = np.sort(stacked, axis=0)

print(
    "Crossing rămas:",
    np.any(np.diff(sorted_values, axis=0) < 0, axis=0).mean()
)

=== ANALIZA CROSSING ===
Total observații: 15708
Q05 > Q50: 13.11%
Q50 > Q95: 13.82%
Q05 > Q95: 0.48%

=== EFECTUL SORTĂRII ASUPRA Q05 ===
Observații modificate: 13.41%
Reducere medie Q05: 0.601 A
Reducere maximă Q05: 41.141 A

=== VERIFICARE ORDINE DUPĂ SORTARE ===
Crossing rămas: 0.0


In [109]:
import numpy as np
import joblib
from pathlib import Path

# Predicții cu modelul antrenat
pred = ml.predict_quantiles(bundle["models"], X_test)

# Verificări
assert np.all(np.isfinite(pred["q05"]))
assert np.all(np.isfinite(pred["q50"]))
assert np.all(np.isfinite(pred["q95"]))

assert np.all(pred["q05"] <= pred["q50"])
assert np.all(pred["q50"] <= pred["q95"])
assert np.all(pred["q05"] >= 0)

print("=== VALIDARE FINALĂ ===")
print("Predicții valide:", len(pred["q05"]))
print("Ordine cuantile: OK")
print("Valori finite: OK")
print("Q05 nenegativ: OK")

print("\n=== STATISTICI PREDICȚII ===")
for name in ["q05", "q50", "q95"]:
    print(
        name,
        "min:", round(float(pred[name].min()), 2),
        "mean:", round(float(pred[name].mean()), 2),
        "max:", round(float(pred[name].max()), 2)
    )

# Salvarea bundle-ului
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "ampacity_model.pkl"
joblib.dump(bundle, model_path, compress=3)

# Testăm și încărcarea modelului salvat
loaded_bundle = joblib.load(model_path)

print("\nModel salvat și reîncărcat:", model_path)
print("Modele:", list(loaded_bundle["models"].keys()))

=== VALIDARE FINALĂ ===
Predicții valide: 15708
Ordine cuantile: OK
Valori finite: OK
Q05 nenegativ: OK

=== STATISTICI PREDICȚII ===
q05 min: 607.64 mean: 1315.12 max: 1637.99
q50 min: 639.14 mean: 1326.67 max: 1766.27
q95 min: 1100.35 mean: 1356.42 max: 1787.49

Model salvat și reîncărcat: models/ampacity_model.pkl
Modele: ['hgb_q05', 'hgb_q50', 'hgb_q95', 'q05_shift_a']


In [110]:
import subprocess

def git(*args):
    result = subprocess.run(
        ["git", *args],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

print("=== BRANCH CURENT ===")
git("branch", "--show-current")

print("\n=== STATUS ===")
git("status", "--short")

print("\n=== MODIFICĂRI LOCALE ===")
git("diff", "--stat")

print("\n=== MODIFICĂRI STAGED ===")
git("diff", "--cached", "--stat")

print("\n=== BRANCH-URI ȘI COMMIT-URI ===")
git("log", "--oneline", "--decorate", "--graph", "--all", "-12")

=== BRANCH CURENT ===
ml-engineer-ampacity


=== STATUS ===
M  data/processed/dataset.csv
M  src/data_pipeline.py
 M src/ml_model.py
?? data/processed/ampacity_test.csv
?? data/processed/ampacity_training.csv
?? data/processed/dataset_before_solar.csv
?? models/ampacity_model.pkl
?? physics_engine_conflict_backup.py


=== MODIFICĂRI LOCALE ===
 gridpilot-ai/src/ml_model.py | 19 +++++++++++++++++--
 1 file changed, 17 insertions(+), 2 deletions(-)


=== MODIFICĂRI STAGED ===
 gridpilot-ai/data/processed/dataset.csv | 210332 ++++++++++++++---------------
 gridpilot-ai/src/data_pipeline.py       |      2 +
 2 files changed, 105168 insertions(+), 105166 deletions(-)


=== BRANCH-URI ȘI COMMIT-URI ===
*-.   3de9ba3 (refs/stash) On main: ML Engineer ampacity work
|\ \  
| | * e372940 untracked files on main: 1897616 Merge remote-tracking branch 'origin/main'
| * 75dae18 index on main: 1897616 Merge remote-tracking branch 'origin/main'
|/  
| * a547c75 (HEAD -> ml-engineer-ampacity, origin/ml

0

In [111]:
import subprocess

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

print("=== FETCH DE PE GITHUB ===")
run_git("fetch", "origin")

print("\n=== COMMIT-URI NOI PE MAIN ===")
run_git("log", "--oneline", "HEAD..origin/main")

print("\n=== STASH-URI EXISTENTE ===")
run_git("stash", "list")

print("\n=== CONȚINUTUL STASH-ULUI PRINCIPAL ===")
run_git("stash", "show", "--stat", "stash@{0}")

print("\n=== MODIFICĂRI STAGED ===")
run_git("diff", "--cached", "--", "src/data_pipeline.py")

print("\n=== MODIFICĂRI LOCALE ML ===")
run_git("diff", "--", "src/ml_model.py")

print("\n=== FIȘIERE STAGED ===")
run_git("diff", "--cached", "--name-status")

print("\n=== FIȘIERE NEURMĂRITE ===")
run_git("ls-files", "--others", "--exclude-standard")

=== FETCH DE PE GITHUB ===

From https://github.com/Fandrei19/Smarthack
 * [new branch]      feature/frontend   -> origin/feature/frontend
   f484b5e..b4e8b86  feature/ml-physics -> origin/feature/ml-physics
   a547c75..18c3ee8  main               -> origin/main


=== COMMIT-URI NOI PE MAIN ===
18c3ee8 tested tge validation fallback
23a53cb implement prediction data validation


=== STASH-URI EXISTENTE ===
stash@{0}: On main: ML Engineer ampacity work


=== CONȚINUTUL STASH-ULUI PRINCIPAL ===
 gridpilot-ai/data/processed/dataset.csv | 210332 ++++++++++++++---------------
 gridpilot-ai/src/data_pipeline.py       |      2 +
 gridpilot-ai/src/physics_engine.py      |    311 +
 3 files changed, 105479 insertions(+), 105166 deletions(-)


=== MODIFICĂRI STAGED ===
diff --git a/gridpilot-ai/src/data_pipeline.py b/gridpilot-ai/src/data_pipeline.py
index afd3d03..766346d 100644
--- a/gridpilot-ai/src/data_pipeline.py
+++ b/gridpilot-ai/src/data_pipeline.py
@@ -30,6 +30,8 @@ FIELDS = {
     "wi

CompletedProcess(args=['git', 'ls-files', '--others', '--exclude-standard'], returncode=0, stdout='data/processed/ampacity_test.csv\ndata/processed/ampacity_training.csv\ndata/processed/dataset_before_solar.csv\nmodels/ampacity_model.pkl\nphysics_engine_conflict_backup.py\n', stderr='')

In [112]:
print("=== ISTORIC ACTUALIZAT ===")
run_git("log", "--oneline", "--decorate", "--graph", "--all", "-20")

print("\n=== MODIFICĂRI FRONTEND FAȚĂ DE MAIN ===")
run_git("diff", "--stat", "origin/main...origin/feature/frontend")

print("\n=== FIȘIERE FRONTEND ===")
run_git("diff", "--name-status", "origin/main...origin/feature/frontend")

print("\n=== DETALII COMMIT-URI NOI MAIN ===")
run_git("show", "--stat", "--oneline", "origin/main", "origin/main~1")

print("\n=== MODIFICĂRI PHYSICS ENGINE DIN STASH ===")
run_git("stash", "show", "-p", "stash@{0}", "--", "gridpilot-ai/src/physics_engine.py")

=== ISTORIC ACTUALIZAT ===
* 18c3ee8 (origin/main, origin/HEAD) tested tge validation fallback
* 23a53cb implement prediction data validation
* a547c75 (HEAD -> ml-engineer-ampacity, origin/ml-engineer-ampacity, main) implement risk-aware decision engine
* a866770 implement multi-point line monitoring
*   67394f6 Merge pull request #1 from Fandrei19/feature/ml-physics
|\  
| * b4e8b86 (origin/feature/ml-physics) Add trained ampacity model and predictions
| *   ac2d157 Merge remote-tracking branch 'origin/main' into feature/ml-physics
| |\  
| |/  
|/|   
* | 20378fd deleted files
| *   f484b5e Merge branch 'main' into feature/ml-physics ase enter a commit message to explain why this merge is necessary,
| |\  
| * | 27289aa WIP
| * | 7c614e6 Add physics engine, ML model, evaluation and tests (Participant 2)
| | | *   3de9ba3 (refs/stash) On main: ML Engineer ampacity work
| |_|/|\  
|/| | | | 
| | | | * e372940 untracked files on main: 1897616 Merge remote-tracking branch 'origin/main'


CompletedProcess(args=['git', 'stash', 'show', '-p', 'stash@{0}', '--', 'gridpilot-ai/src/physics_engine.py'], returncode=1, stdout='', stderr="Too many revisions specified: 'stash@{0}' 'gridpilot-ai/src/physics_engine.py'\n")

In [113]:
import subprocess

def git_safe(*args):
    result = subprocess.run(
        ["git", *args],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Comanda git a eșuat: {args}")
    return result

# 1. Salvăm toate modificările locale, inclusiv fișierele noi
git_safe(
    "stash", "push", "-u",
    "-m", "backup before integrating latest main"
)

# 2. Verificăm că directorul de lucru este curat
git_safe("status", "--short")

# 3. Actualizăm branch-ul cu cele două commit-uri ale colegilor
git_safe("merge", "--ff-only", "origin/main")

# 4. Recuperăm modificările noastre, fără să ștergem stash-ul
git_safe("stash", "apply", "stash@{0}")

# 5. Verificăm rezultatul
git_safe("status", "--short")

Saved working directory and index state On ml-engineer-ampacity: backup before integrating latest main


Updating a547c75..18c3ee8
Fast-forward
 gridpilot-ai/src/data_validator.py                 | 145 +++++++++++++++++++++
 gridpilot-ai/tests/test_data_validator.py          |  53 ++++++++
 .../tests/test_data_validator_duplicates.py        |  45 +++++++
 .../tests/test_data_validator_edge_cases.py        |  58 +++++++++
 gridpilot-ai/tests/test_data_validator_quality.py  |  59 +++++++++
 gridpilot-ai/tests/test_predictions_csv.py         |  15 +++
 gridpilot-ai/tests/test_validation_fallback.py     |  38 ++++++
 7 files changed, 413 insertions(+)
 create mode 100644 gridpilot-ai/src/data_validator.py
 create mode 100644 gridpilot-ai/tests/test_data_validator.py
 create mode 100644 gridpilot-ai/tests/test_data_validator_duplicates.py
 create mode 100644 gridpilot-ai/tests/test_data_validator_edge_cases.py
 create mode 100644 gridpilot-ai/tests/test_data_validator_quality.py
 create mod

CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout=' M data/processed/dataset.csv\n M src/data_pipeline.py\n M src/ml_model.py\n?? data/processed/ampacity_test.csv\n?? data/processed/ampacity_training.csv\n?? data/processed/dataset_before_solar.csv\n?? models/ampacity_model.pkl\n?? physics_engine_conflict_backup.py\n', stderr='')

In [114]:
import os

print("=== NOTEBOOK-URI ===")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".git", "site-packages"]]
    for f in files:
        if f.endswith(".ipynb") and "Smarthack" in root:
            path = os.path.join(root, f)
            print(path, f"{os.path.getsize(path)/1024:.1f} KB")

print("\n=== DIMENSIUNI FIȘIERE ===")
files = [
    "data/processed/ampacity_training.csv",
    "data/processed/ampacity_test.csv",
    "data/processed/dataset.csv",
    "models/ampacity_model.pkl",
]

for f in files:
    if os.path.exists(f):
        print(f, f"{os.path.getsize(f)/1024/1024:.2f} MB")

=== NOTEBOOK-URI ===
/content/Smarthack/gridpilot-ai/notebooks/01_data_exploration.ipynb 0.0 KB
/content/Smarthack/gridpilot-ai/notebooks/02_model_training.ipynb 0.0 KB

=== DIMENSIUNI FIȘIERE ===
data/processed/ampacity_training.csv 20.07 MB
data/processed/ampacity_test.csv 0.00 MB
data/processed/dataset.csv 23.63 MB
models/ampacity_model.pkl 1.40 MB
